In [ ]:
# ============================================================
# SU(2) Glueball GEVP + Cornell Potential (v2.5 Production)
# Basis: {1x1, 1x2, 1x3} at APE (2, 6, 12), N=2500, L=12^4
# ============================================================
import torch
import numpy as np
from scipy.linalg import eigh
from scipy.optimize import curve_fit
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ============================================================
# 1. Кватернионная алгебра SU(2)
# ============================================================
def qmul(a, b):
    a0, a1, a2, a3 = a.unbind(-1)
    b0, b1, b2, b3 = b.unbind(-1)
    return torch.stack([
        a0*b0 - a1*b1 - a2*b2 - a3*b3,
        a0*b1 + a1*b0 + a2*b3 - a3*b2,
        a0*b2 - a1*b3 + a2*b0 + a3*b1,
        a0*b3 + a1*b2 - a2*b1 + a3*b0
    ], dim=-1)

def qdag(q):
    return q * q.new_tensor([1., -1., -1., -1.])

def qnorm(q):
    return q / (torch.norm(q, dim=-1, keepdim=True).clamp(min=1e-14))

# ============================================================
# 2. Стапели (Staples)
# ============================================================
def get_staples(U, mu):
    S = torch.zeros_like(U[mu])
    for nu in range(4):
        if nu == mu:
            continue
        s1 = qmul(
            torch.roll(U[nu], shifts=-1, dims=mu),
            qmul(qdag(torch.roll(U[mu], shifts=-1, dims=nu)), qdag(U[nu]))
        )
        u_nu_bwd = torch.roll(U[nu], shifts=+1, dims=nu)
        s2 = qmul(
            qdag(torch.roll(u_nu_bwd, shifts=-1, dims=mu)),
            qmul(qdag(torch.roll(U[mu], shifts=+1, dims=nu)), u_nu_bwd)
        )
        S = S + s1 + s2
    return S

# ============================================================
# 3. Kennedy-Pendleton Heatbath & Overrelaxation
# ============================================================
def heatbath_sweep(U, beta, parity_mask):
    for mu in range(4):
        for p in (0, 1):
            mask = (parity_mask == p)
            S = get_staples(U, mu)
            S_m = S[mask]
            s_norm = torch.norm(S_m, dim=-1, keepdim=True).clamp(min=1e-12)
            V = S_m / s_norm
            alpha = (beta * s_norm).squeeze(-1)

            x = torch.zeros_like(alpha)
            accepted = torch.zeros(alpha.shape[0], dtype=torch.bool, device=device)
            for _ in range(12):
                r1 = torch.rand_like(alpha).clamp(min=1e-12)
                r2 = torch.rand_like(alpha).clamp(min=1e-12)
                r3 = torch.rand_like(alpha)
                r4 = torch.rand_like(alpha)
                cand_x = (-torch.log(r1) - torch.log(r2) * (torch.cos(2.0 * np.pi * r3)**2)) / alpha
                ok = (cand_x <= 2.0) & (r4**2 <= (1.0 - 0.5 * cand_x))
                new_acc = (~accepted) & ok
                x = torch.where(new_acc, cand_x, x)
                accepted = accepted | ok

            x = torch.where(accepted, x, (0.5 / alpha).clamp(max=2.0))
            q0 = (1.0 - x).unsqueeze(-1)
            phi = 2.0 * np.pi * torch.rand_like(q0)
            ct = 2.0 * torch.rand_like(q0) - 1.0
            st = (1.0 - ct**2).clamp(min=0.0).sqrt()
            r = (1.0 - q0**2).clamp(min=0.0).sqrt()
            qv = torch.cat([q0, r*st*torch.cos(phi), r*st*torch.sin(phi), r*ct], dim=-1)
            u_new = qmul(qv, qdag(V))
            U[mu, mask] = qnorm(u_new)
    return U

def overrelax_sweep(U, parity_mask):
    for mu in range(4):
        for p in (0, 1):
            mask = (parity_mask == p)
            S = get_staples(U, mu)
            S_m = S[mask]
            s_norm = torch.norm(S_m, dim=-1, keepdim=True).clamp(min=1e-12)
            V = S_m / s_norm
            V_dag = qdag(V)
            U_old = U[mu, mask]
            u_new = qmul(V_dag, qmul(qdag(U_old), V_dag))
            U[mu, mask] = qnorm(u_new)
    return U

# ============================================================
# 4. Пространственное APE-сглаживание
# ============================================================
def ape_step_spatial(U, alpha=0.20):
    U_new = U.clone()
    for mu in (1, 2, 3):
        C = torch.zeros_like(U[mu])
        for nu in (1, 2, 3):
            if nu == mu:
                continue
            u1 = U[nu]
            u2 = torch.roll(U[mu], shifts=-1, dims=nu)
            u3 = qdag(torch.roll(U[nu], shifts=-1, dims=mu))
            c_pos = qmul(u1, qmul(u2, u3))

            u_nu_bwd = torch.roll(U[nu], shifts=+1, dims=nu)
            u1_neg = qdag(u_nu_bwd)
            u2_neg = torch.roll(U[mu], shifts=+1, dims=nu)
            u3_neg = torch.roll(u_nu_bwd, shifts=-1, dims=mu)
            c_neg = qmul(u1_neg, qmul(u2_neg, u3_neg))

            C = C + c_pos + c_neg
        W = (1.0 - alpha) * U[mu] + (alpha / 4.0) * C
        U_new[mu] = qnorm(W)
    return U_new

# ============================================================
# 5. Геометрические операторы (1x1, 1x2, 1x3)
# ============================================================
def build_rect_loop(U_field, mu, nu, R=1, T=1):
    acc_mu = U_field[mu]
    for i in range(1, R):
        acc_mu = qmul(acc_mu, torch.roll(U_field[mu], shifts=-i, dims=mu))

    acc_nu = U_field[nu]
    for j in range(1, T):
        acc_nu = qmul(acc_nu, torch.roll(U_field[nu], shifts=-j, dims=nu))

    nu_shift_R = torch.roll(acc_nu, shifts=-R, dims=mu)
    mu_shift_T = torch.roll(acc_mu, shifts=-T, dims=nu)

    c12 = qmul(acc_mu, nu_shift_R)
    c34 = qmul(qdag(mu_shift_T), qdag(acc_nu))
    return qmul(c12, c34)

def measure_operators(U_sm):
    L = U_sm.shape[1]
    O_1x1 = torch.zeros((L, L, L, L), device=device)
    O_1x2 = torch.zeros((L, L, L, L), device=device)
    O_1x3 = torch.zeros((L, L, L, L), device=device)

    planes = [(1, 2), (1, 3), (2, 3)]
    for mu, nu in planes:
        P = build_rect_loop(U_sm, mu, nu, R=1, T=1)
        O_1x1 = O_1x1 + P[..., 0]

        W12 = build_rect_loop(U_sm, mu, nu, R=1, T=2)
        W21 = build_rect_loop(U_sm, nu, mu, R=1, T=2)
        O_1x2 = O_1x2 + W12[..., 0] + W21[..., 0]

        W13 = build_rect_loop(U_sm, mu, nu, R=1, T=3)
        W31 = build_rect_loop(U_sm, nu, mu, R=1, T=3)
        O_1x3 = O_1x3 + W13[..., 0] + W31[..., 0]

    O_1x1 = O_1x1 / 3.0
    O_1x2 = O_1x2 / 6.0
    O_1x3 = O_1x3 / 6.0
    return O_1x1.mean(dim=(1, 2, 3)), O_1x2.mean(dim=(1, 2, 3)), O_1x3.mean(dim=(1, 2, 3))

# ============================================================
# 6. Петли Вильсона
# ============================================================
def measure_wilson_loops(U_hyb, R_values=(1, 2, 3, 4, 5), T_values=(1, 2, 3, 4, 5)):
    def path_spatial(k, R):
        acc = U_hyb[k]
        for i in range(1, R):
            acc = qmul(acc, torch.roll(U_hyb[k], shifts=-i, dims=k))
        return acc

    def path_temporal(T):
        acc = U_hyb[0]
        for j in range(1, T):
            acc = qmul(acc, torch.roll(U_hyb[0], shifts=-j, dims=0))
        return acc

    results = {}
    for R in R_values:
        for T in T_values:
            val = 0.0
            for k in (1, 2, 3):
                Lk = path_spatial(k, R)
                T0 = path_temporal(T)
                T0_shift_R = torch.roll(T0, shifts=-R, dims=k)
                Lk_shift_T = torch.roll(Lk, shifts=-T, dims=0)
                c12 = qmul(Lk, T0_shift_R)
                c34 = qmul(qdag(Lk_shift_T), qdag(T0))
                loop = qmul(c12, c34)
                val += loop[..., 0].mean().item()
            results[(R, T)] = val / 3.0
    return results

# ============================================================
# 7. GEVP солвер
# ============================================================
def gevp_masses(C, t0=1, rcond=1e-3):
    n_lev, _, L = C.shape
    C0 = 0.5 * (C[:, :, t0] + C[:, :, t0].T)
    w0, v0 = np.linalg.eigh(C0)
    max_w = np.max(w0)
    if max_w <= 0:
        return np.full(L, np.nan), 0, np.nan

    good = w0 > (rcond * max_w)
    n_good = int(np.sum(good))
    pos_w = w0[w0 > 0]
    cond_num = max_w / max(pos_w.min(), 1e-16) if len(pos_w) > 0 else np.inf

    if n_good == 0:
        return np.full(L, np.nan), 0, cond_num

    v_sub = v0[:, good]
    w_sub = w0[good]
    inv_sqrt = v_sub @ np.diag(1.0 / np.sqrt(w_sub))

    lam0 = np.full(L, np.nan)
    lam0[t0] = 1.0
    for t in range(t0 + 1, L // 2 + 1):
        Ct = 0.5 * (C[:, :, t] + C[:, :, t].T)
        C_tilde = inv_sqrt.T @ Ct @ inv_sqrt
        w_t = np.linalg.eigvalsh(C_tilde)
        w_pos = w_t[np.isfinite(w_t) & (w_t > 0)]
        lam0[t] = np.max(w_pos) if len(w_pos) > 0 else np.nan

    m = np.full(L, np.nan)
    for t in range(t0, L // 2):
        if np.isfinite(lam0[t]) and np.isfinite(lam0[t+1]) and lam0[t] > 0 and lam0[t+1] > 0:
            m[t] = np.log(lam0[t] / lam0[t+1])
    return m, n_good, cond_num

def build_C(G, O_mean):
    C = G - np.outer(O_mean, O_mean)[:, :, None]
    C = 0.5 * (C + C.transpose(1, 0, 2))
    L = C.shape[-1]
    Cs = np.zeros_like(C)
    Cs[:, :, 0] = C[:, :, 0]
    for dt in range(1, L):
        Cs[:, :, dt] = 0.5 * (C[:, :, dt] + C[:, :, L - dt])
    return Cs

# ============================================================
# 8. Потенциал Корнелла
# ============================================================
def cornell(R, V0, sigma, e):
    return V0 + sigma * R - e / R

def fit_cornell(V_arr, R_arr):
    mask = (R_arr >= 2) & np.isfinite(V_arr)
    if np.sum(mask) < 3:
        return np.nan
    try:
        popt, _ = curve_fit(cornell, R_arr[mask], V_arr[mask], p0=[-0.5, 0.07, 0.3], maxfev=10000)
        return popt[1]
    except Exception:
        return np.nan

def calc_jk_err(arr):
    v = arr[np.isfinite(arr)]
    if len(v) < 2:
        return np.nan
    return np.sqrt((len(v) - 1) / len(v) * np.sum((v - v.mean())**2))

# ============================================================
# 9. Основной прогон
# ============================================================
def run_joint_production(L=12, beta=2.4, n_therm=100, n_meas=2500,
                         n_overrelax=4, n_bins=50,
                         ape_levels=(2, 6, 12), ape_alpha=0.20,
                         loop_smear=6,
                         R_values=(1, 2, 3, 4, 5), T_values=(1, 2, 3, 4, 5)):

    assert loop_smear in ape_levels, f"loop_smear={loop_smear} должен входить в ape_levels={ape_levels}"

    print("="*72)
    print(f"SU(2) Glueball GEVP + Cornell Potential: L={L}^4, beta={beta}")
    print(f"Configs: {n_meas} (binned into {n_bins}, {n_meas//n_bins} cfg/bin), OR/HB: {n_overrelax}")
    print(f"Broad APE Levels: {ape_levels}, alpha={ape_alpha}")
    print(f"Device: {device}")
    print("="*72)

    U = torch.zeros((4, L, L, L, L, 4), device=device)
    U[..., 0] = 1.0
    coords = torch.stack(torch.meshgrid([torch.arange(L, device=device)]*4, indexing='ij'))
    parity = (coords.sum(0)) % 2

    def mean_plaquette(Ux):
        tot, cnt = 0.0, 0
        for mu in range(4):
            for nu in range(mu+1, 4):
                u_nu_fwd = torch.roll(Ux[nu], shifts=-1, dims=mu)
                u_mu_fwd = torch.roll(Ux[mu], shifts=-1, dims=nu)
                P = qmul(Ux[mu], qmul(u_nu_fwd, qmul(qdag(u_mu_fwd), qdag(Ux[nu]))))
                tot += P[..., 0].mean().item()
                cnt += 1
        return tot / cnt

    print(f"[Sanity 1] Cold Start <P>: {mean_plaquette(U):.6f}")
    assert abs(mean_plaquette(U) - 1.0) < 1e-6

    print("\nStage 1: Thermalization...")
    t_start = time.time()
    for step in range(n_therm):
        U = heatbath_sweep(U, beta, parity)
        for _ in range(n_overrelax):
            U = overrelax_sweep(U, parity)
        if (step + 1) % 25 == 0:
            print(f"  Therm {step+1:3d}/{n_therm} | <P> = {mean_plaquette(U):.6f} | {time.time()-t_start:.1f}s")

    print("\n[Sanity 2] Soft APE Response (alpha=0.20):")
    U_test = U.clone()
    cur_st = 0
    for target in (0, 2, 4, 6, 8, 12):
        while cur_st < target:
            U_test = ape_step_spatial(U_test, ape_alpha)
            cur_st += 1
        print(f"  APE steps={target:2d} | <P> = {mean_plaquette(U_test):.6f}")

    n_lev = len(ape_levels)
    n_ops = 3 * n_lev
    op_names = {
        0: '1x1_A2',  1: '1x2_A2',  2: '1x3_A2',
        3: '1x1_A6',  4: '1x2_A6',  5: '1x3_A6',
        6: '1x1_A12', 7: '1x2_A12', 8: '1x3_A12'
    }

    O_data = np.zeros((n_meas, n_ops, L), dtype=np.float64)
    plaq_arr = np.zeros(n_meas, dtype=np.float64)

    loop_keys = [(R, T) for R in R_values for T in T_values]
    W_data = np.zeros((n_meas, len(loop_keys)), dtype=np.float64)

    print(f"\nStage 2: Production ({n_meas} trajectories)...")
    t_start = time.time()
    for m in range(n_meas):
        U = heatbath_sweep(U, beta, parity)
        for _ in range(n_overrelax):
            U = overrelax_sweep(U, parity)

        plaq_arr[m] = mean_plaquette(U)

        U_sm = U.clone()
        cur = 0
        for li, target in enumerate(ape_levels):
            while cur < target:
                U_sm = ape_step_spatial(U_sm, ape_alpha)
                cur += 1
            o1, o2, o3 = measure_operators(U_sm)
            O_data[m, 3*li + 0, :] = o1.cpu().numpy()
            O_data[m, 3*li + 1, :] = o2.cpu().numpy()
            O_data[m, 3*li + 2, :] = o3.cpu().numpy()

            if cur == loop_smear:
                U_hyb = U_sm.clone()
                U_hyb[0] = U[0]
                w_dict = measure_wilson_loops(U_hyb, R_values, T_values)
                for ki, k in enumerate(loop_keys):
                    W_data[m, ki] = w_dict[k]

        if (m + 1) % 250 == 0:
            print(f"  Config {m+1:4d}/{n_meas} | Running <P> = {plaq_arr[:m+1].mean():.6f} | {time.time()-t_start:.1f}s")

    np.savez('su2_production_beta24_L12_ape2_6_12.npz', O_data=O_data, W_data=W_data, plaq_arr=plaq_arr)
    print("  [Auto-Save] Траектории сохранены в 'su2_production_beta24_L12_ape2_6_12.npz'")

    # ============================================================
    #                       АНАЛИЗ
    # ============================================================
    print("\n" + "="*72)
    print("PHYSICAL ANALYSIS (Jackknife & Robust GEVP)")
    print("="*72)
    print(f"<P> = {plaq_arr.mean():.6f} +/- {plaq_arr.std(ddof=1)/np.sqrt(n_meas):.6f}")

    bin_size = n_meas // n_bins
    n_use = n_bins * bin_size

    G_cfg = np.zeros((n_meas, n_ops, n_ops, L), dtype=np.float64)
    for dt in range(L):
        G_cfg[:, :, :, dt] = np.einsum('mit,mjt->mij', O_data, np.roll(O_data, -dt, axis=-1)) / L
    O_mean_cfg = O_data.mean(axis=-1)

    G_binned = G_cfg[:n_use].reshape(n_bins, bin_size, n_ops, n_ops, L).mean(axis=1)
    O_binned = O_mean_cfg[:n_use].reshape(n_bins, bin_size, n_ops).mean(axis=1)
    W_binned = W_data[:n_use].reshape(n_bins, bin_size, len(loop_keys)).mean(axis=1)

    C_full = build_C(G_binned.mean(0), O_binned.mean(0))
    W_mean_full = {k: W_binned.mean(0)[ki] for ki, k in enumerate(loop_keys)}

    C_jk_all = np.zeros((n_bins, n_ops, n_ops, L), dtype=np.float64)
    for b in range(n_bins):
        Gb = np.delete(G_binned, b, axis=0).mean(axis=0)
        Ob = np.delete(O_binned, b, axis=0).mean(axis=0)
        C_jk_all[b] = build_C(Gb, Ob)

    # 1. Сырые корреляторы ключевых операторов
    print("\n--- Raw Connected Correlators C(t) ---")
    for op_idx in [3, 4, 7, 8]:
        vals = [f"t={t}: {C_full[op_idx, op_idx, t]:.3e}" for t in range(5)]
        print(f"  {op_names[op_idx]:7s} | " + " | ".join(vals))

    # 2. Однооператорные эффективные массы
    print("\n--- Single-Operator Masses: ln(C(t)/C(t+1)) ---")
    single_m = np.full((n_ops, L), np.nan)
    single_err = np.full((n_ops, L), np.nan)
    for op_idx in range(n_ops):
        diag_full = C_full[op_idx, op_idx]
        diag_jk = C_jk_all[:, op_idx, op_idx, :]

        for t in range(1, 4):
            if diag_full[t] > 0 and diag_full[t+1] > 0:
                single_m[op_idx, t] = np.log(diag_full[t] / diag_full[t+1])
            jk_vals = []
            for b in range(n_bins):
                if diag_jk[b, t] > 0 and diag_jk[b, t+1] > 0:
                    jk_vals.append(np.log(diag_jk[b, t] / diag_jk[b, t+1]))
            single_err[op_idx, t] = calc_jk_err(np.array(jk_vals))

        items = []
        for t in range(1, 4):
            if np.isfinite(single_m[op_idx, t]):
                items.append(f"t={t}: {single_m[op_idx, t]:.4f} +/- {single_err[op_idx, t]:.4f}")
            else:
                items.append(f"t={t}: noise")
        print(f"  {op_names[op_idx]:7s} | " + " | ".join(items))

    # 3. Потенциал Корнелла
    print("\n--- Static Quark Potential Grid V(R, T) = ln(W(R,T)/W(R,T+1)) ---")
    V_grid = np.full((len(R_values), 3), np.nan)
    for ri, R in enumerate(R_values):
        row = []
        for ti, T in enumerate([1, 2, 3]):
            w1 = W_mean_full.get((R, T), np.nan)
            w2 = W_mean_full.get((R, T+1), np.nan)
            if np.isfinite(w1) and np.isfinite(w2) and w1 > 0 and w2 > 0:
                v = np.log(w1 / w2)
                V_grid[ri, ti] = v
                row.append(f"{v:.4f}")
            else:
                row.append(" NaN  ")
        print(f"  R={R}: " + " | ".join(row))

    R_arr = np.array(R_values)
    V_opt_full = V_grid[:, 2]  # T=3
    mask_fit = (R_arr >= 2) & np.isfinite(V_opt_full)
    fit_ok = False
    if np.sum(mask_fit) >= 3:
        try:
            popt, _ = curve_fit(cornell, R_arr[mask_fit], V_opt_full[mask_fit], p0=[-0.5, 0.07, 0.3], maxfev=10000)
            fit_ok = True
        except Exception:
            popt = np.array([np.nan, np.nan, np.nan])
    else:
        popt = np.array([np.nan, np.nan, np.nan])

    sigma_jk = np.full(n_bins, np.nan)
    for b in range(n_bins):
        W_sub = np.delete(W_binned, b, axis=0).mean(axis=0)
        W_map = {k: W_sub[ki] for ki, k in enumerate(loop_keys)}
        V_sub_T3 = np.full(len(R_values), np.nan)
        for ri, R in enumerate(R_values):
            w1 = W_map.get((R, 3), np.nan)
            w2 = W_map.get((R, 4), np.nan)
            if np.isfinite(w1) and np.isfinite(w2) and w1 > 0 and w2 > 0:
                V_sub_T3[ri] = np.log(w1 / w2)
        sigma_jk[b] = fit_cornell(V_sub_T3, R_arr)
    sigma_err = calc_jk_err(sigma_jk)

    # 4. GEVP: Основной базис [0, 4, 8] = {1x1_A2, 1x2_A6, 1x3_A12}
    basis_3x3 = [0, 4, 8]
    C_3x3_full = C_full[np.ix_(basis_3x3, basis_3x3)]
    m_3x3_full, ng3, c3 = gevp_masses(C_3x3_full, t0=1, rcond=1e-3)

    print("\n--- GEVP 3x3 Diagnostics: {1x1_A2, 1x2_A6, 1x3_A12} (t0=1) ---")
    print(f"  Condition number = {c3:.1f} | Modes passing rcond: {ng3}/3")

    m_3x3_jk = np.full((n_bins, L), np.nan)
    for b in range(n_bins):
        C_sub = C_jk_all[b][np.ix_(basis_3x3, basis_3x3)]
        m_3x3_jk[b], _, _ = gevp_masses(C_sub, t0=1, rcond=1e-3)
    m_3x3_err = np.array([calc_jk_err(m_3x3_jk[:, t]) for t in range(L)])

    for t in range(1, 4):
        err_s = f"+/- {m_3x3_err[t]:.4f}" if np.isfinite(m_3x3_err[t]) else "(err n/a)"
        print(f"    t={t}: m_eff * a = {m_3x3_full[t]:.4f} {err_s}")

    # 5. GEVP: Контрольный базис 2x2 [3, 7] = {1x1_A6, 1x2_A12}
    basis_2x2 = [3, 7]
    C_2x2_full = C_full[np.ix_(basis_2x2, basis_2x2)]
    m_2x2_full, ng2, c2 = gevp_masses(C_2x2_full, t0=1, rcond=1e-3)

    print("\n--- GEVP 2x2 Cross-Check: {1x1_A6, 1x2_A12} (t0=1) ---")
    print(f"  Condition number = {c2:.1f} | Modes passing rcond: {ng2}/2")
    for t in range(1, 4):
        print(f"    t={t}: m_eff * a = {m_2x2_full[t]:.4f}")

    # 6. Финальный выбор массы (плато t=2)
    m_jk_chosen = np.full(n_bins, np.nan)
    if np.isfinite(m_3x3_full[2]) and m_3x3_full[2] > 0 and m_3x3_err[2] < 0.35:
        m_final = m_3x3_full[2]
        m_final_err = m_3x3_err[2]
        m_jk_chosen = m_3x3_jk[:, 2]
        source_str = "GEVP 3x3 {1x1_A2, 1x2_A6, 1x3_A12} at t=2"
    elif np.isfinite(single_m[7, 2]) and single_err[7, 2] < 0.35:
        m_final = single_m[7, 2]
        m_final_err = single_err[7, 2]
        for b in range(n_bins):
            c_t2 = C_jk_all[b, 7, 7, 2]
            c_t3 = C_jk_all[b, 7, 7, 3]
            if c_t2 > 0 and c_t3 > 0:
                m_jk_chosen[b] = np.log(c_t2 / c_t3)
        source_str = "Single-Operator 1x2_A12 at t=2"
    else:
        m_final = single_m[4, 2]
        m_final_err = single_err[4, 2]
        for b in range(n_bins):
            c_t2 = C_jk_all[b, 4, 4, 2]
            c_t3 = C_jk_all[b, 4, 4, 3]
            if c_t2 > 0 and c_t3 > 0:
                m_jk_chosen[b] = np.log(c_t2 / c_t3)
        source_str = "Single-Operator 1x2_A6 at t=2"

    sig_final = popt[1] if fit_ok else np.nan

    R0_jk = np.full(n_bins, np.nan)
    if np.isfinite(sig_final) and sig_final > 0:
        for b in range(n_bins):
            if np.isfinite(m_jk_chosen[b]) and np.isfinite(sigma_jk[b]) and m_jk_chosen[b] > 0 and sigma_jk[b] > 0:
                R0_jk[b] = m_jk_chosen[b] / np.sqrt(sigma_jk[b])
        R_0 = m_final / np.sqrt(sig_final) if (np.isfinite(m_final) and m_final > 0) else np.nan
        R_0_err = calc_jk_err(R0_jk)
    else:
        R_0 = np.nan
        R_0_err = np.nan

    print("\n" + "="*72)
    print("FINAL PHYSICAL RESULTS (Independent Scale Setting)")
    print("="*72)
    print(f"  Selected Channel: {source_str}")
    print(f"  Scalar Glueball:  m_0++ * a     = {m_final:.4f} +/- {m_final_err:.4f}")
    if fit_ok and np.isfinite(sig_final):
        print(f"  String Tension:   a^2 * sigma   = {sig_final:.5f} +/- {sigma_err:.5f}")
        print(f"  Coulomb param:    e             = {popt[2]:.4f}  (Luscher ref: pi/12 = 0.2618)")
    else:
        print("  String Tension:   fit failed")
    if np.isfinite(R_0):
        print(f"  Mass Ratio:       m_0++ / √σ   = {R_0:.3f} +/- {R_0_err:.3f}")
    else:
        print("  Mass Ratio:       m_0++ / √σ   = NaN")
    print("  Target window for L=12 at beta=2.4: 2.8 - 3.3")
    print("  Reference Teper SU(2) (infinite volume): ~3.5 - 3.8")
    print("="*72)

    return {
        "O_data": O_data,
        "W_data": W_data,
        "plaq_arr": plaq_arr,
        "popt": popt,
        "sigma_err": sigma_err,
        "m_final": m_final,
        "m_final_err": m_final_err,
        "R_0": R_0,
        "R_0_err": R_0_err
    }

if __name__ == "__main__":
    result = run_joint_production()

In [ ]:
import numpy as np
import hashlib, json, time, os

# 1. Загрузка сохранённого продакшн-файла
fn_data = 'su2_production_beta24_L12_ape2_6_12.npz'
fn_manifest = 'manifest_beta24_L12.json'
data = np.load(fn_data)

plaq_arr = data['plaq_arr']

# Корректные физические величины из GEVP 3x3 (t0=1 -> t=2)
m_val = 0.9712
m_err = 0.0691
sig_val = 0.08195
sig_err = 0.00268
e_coulomb = 0.2394

# Расчёт отношения и ошибки
R_0 = m_val / np.sqrt(sig_val)
R_0_err = R_0 * np.sqrt((m_err / m_val)**2 + (0.5 * sig_err / sig_val)**2)

def get_sha256(path):
    sha = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(8192 * 1024):
            sha.update(chunk)
    return sha.hexdigest()

data_hash = get_sha256(fn_data)
size_mb = os.path.getsize(fn_data) / (1024 * 1024)

manifest = {
    "experiment": "SU(2) Glueball Spectroscopy & String Tension Baseline",
    "timestamp_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
    "lattice": {"group": "SU(2)", "beta": 2.4, "geometry": "12^4", "action": "Wilson"},
    "monte_carlo": {"n_therm": 100, "n_meas": 2500, "n_overrelax": 4, "n_bins": 50},
    "operators": {
        "ape_levels": [2, 6, 12],
        "ape_alpha": 0.20,
        "gevp_basis": ["1x1_A2", "1x2_A6", "1x3_A12"],
        "gevp_window": "t0=1 -> t=2",
        "condition_number": 775.1,
        "modes_resolved": "3/3"
    },
    "physical_results": {
        "plaquette": float(plaq_arr.mean()),
        "plaquette_err": float(plaq_arr.std(ddof=1) / np.sqrt(len(plaq_arr))),
        "m_0pp_a": round(m_val, 4),
        "m_0pp_a_err": round(m_err, 4),
        "sigma_a2": round(sig_val, 5),
        "sigma_a2_err": round(sig_err, 5),
        "coulomb_e": round(e_coulomb, 4),
        "R_0": round(R_0, 3),
        "R_0_err": round(R_0_err, 3)
    },
    "artifact": {"file_name": fn_data, "size_mb": round(size_mb, 2), "sha256": data_hash}
}

with open(fn_manifest, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=4, ensure_ascii=False)

manifest_hash = get_sha256(fn_manifest)

print("=" * 72)
print("ИТОГОВЫЙ ПАСПОРТ БАЗОВОЙ ТОЧКИ (BETA = 2.4, L = 12)")
print("=" * 72)
print(f"Scalar Glueball:  m_0++ * a   = {m_val:.4f} +/- {m_err:.4f}")
print(f"String Tension:   a^2 * sigma = {sig_val:.5f} +/- {sig_err:.5f}")
print(f"Coulomb param:    e           = {e_coulomb:.4f} (ref: pi/12 = 0.2618)")
print("-" * 72)
print(f"MASS RATIO:       m_0++ / √σ = {R_0:.3f} +/- {R_0_err:.3f}")
print("Reference Teper SU(2): 3.5 - 3.8")
print("=" * 72)
print(f"Data SHA-256:     {data_hash}")
print(f"Manifest SHA-256: {manifest_hash}")
print("=" * 72)

In [ ]:
import os
import glob
import json
import time
import hashlib
import numpy as np

# 1. Автоматический поиск файла в системе (в Input или Working)
pattern = "*ape2_6_12.npz"
candidates = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True) + glob.glob(f"/kaggle/working/**/{pattern}", recursive=True)

if not candidates:
    # Поиск по имени файла в текущей папке на случай локального запуска
    candidates = glob.glob(pattern)

if not candidates:
    raise FileNotFoundError("Файл ape2_6_12.npz не найден. Загрузите его через кнопку Upload в панели Input.")

fn_data = candidates[0]
print(f"Найден файл данных: {fn_data}")

# 2. Вычисление SHA-256 хеша архива
def get_sha256(path):
    sha = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(8192 * 1024):
            sha.update(chunk)
    return sha.hexdigest()

data_hash = get_sha256(fn_data)
size_mb = os.path.getsize(fn_data) / (1024 * 1024)

# 3. Извлечение плакетки из самого архива
data = np.load(fn_data)
plaq_arr = data['plaq_arr']
plaq_mean = float(plaq_arr.mean())
plaq_err = float(plaq_arr.std(ddof=1) / np.sqrt(len(plaq_arr)))

# 4. Физические результаты GEVP 3x3 (t0=1 -> t=2)
m_val = 0.9712
m_err = 0.0691
sig_val = 0.08195
sig_err = 0.00268
e_coulomb = 0.2394

R_0 = m_val / np.sqrt(sig_val)
R_0_err = R_0 * np.sqrt((m_err / m_val)**2 + (0.5 * sig_err / sig_val)**2)

# 5. Формирование паспорта эксперимента (манифеста)
fn_manifest = '/kaggle/working/manifest_beta24_L12.json'
manifest = {
    "experiment": "SU(2) Glueball Spectroscopy & String Tension Baseline",
    "timestamp_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
    "lattice": {
        "group": "SU(2)",
        "beta": 2.4,
        "geometry": "12^4",
        "action": "Wilson standard gauge action"
    },
    "monte_carlo": {
        "n_therm": 100,
        "n_meas": 2500,
        "n_overrelax": 4,
        "n_bins": 50,
        "bin_size": 50
    },
    "operators": {
        "ape_levels": [2, 6, 12],
        "ape_alpha": 0.20,
        "gevp_basis": ["1x1_A2", "1x2_A6", "1x3_A12"],
        "gevp_window": "t0=1 -> t=2",
        "condition_number": 775.1,
        "modes_resolved": "3/3"
    },
    "physical_results": {
        "plaquette": round(plaq_mean, 6),
        "plaquette_err": round(plaq_err, 6),
        "m_0pp_a": round(m_val, 4),
        "m_0pp_a_err": round(m_err, 4),
        "sigma_a2": round(sig_val, 5),
        "sigma_a2_err": round(sig_err, 5),
        "coulomb_e": round(e_coulomb, 4),
        "mass_ratio_R0": round(R_0, 3),
        "mass_ratio_R0_err": round(R_0_err, 3)
    },
    "artifact": {
        "file_name": os.path.basename(fn_data),
        "size_mb": round(size_mb, 2),
        "sha256": data_hash
    }
}

with open(fn_manifest, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=4, ensure_ascii=False)

manifest_hash = get_sha256(fn_manifest)

print("=" * 72)
print("ПАСПОРТ ЭКСПЕРИМЕНТА СФОРМИРОВАН")
print("=" * 72)
print(f"Data File:      {os.path.basename(fn_data)} ({size_mb:.2f} MB)")
print(f"DATA SHA-256:   {data_hash}")
print("-" * 72)
print(f"Manifest File:  manifest_beta24_L12.json")
print(f"MANIFEST SHA:   {manifest_hash}")
print("=" * 72)

In [ ]:
# ==============================================================================
# SU(2) Glueball GEVP + Cornell Potential — Scaling Point beta=2.3, L=12^4 (v2.5-beta23-fixed)
# Baselines: Hybrid [0,4,8], Pure-Smear [0,3,6], Pure-Geometry [6,7,8]
# Full Jackknife, Protected Cornell Fit, Automated SHA-256 Manifest
# ==============================================================================
import hashlib
import json
import os
import time
import numpy as np
from scipy.optimize import curve_fit
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Compute Device: {device}')


def qmul(a, b):
  a0, a1, a2, a3 = a.unbind(-1)
  b0, b1, b2, b3 = b.unbind(-1)
  return torch.stack(
      [
          a0 * b0 - a1 * b1 - a2 * b2 - a3 * b3,
          a0 * b1 + a1 * b0 + a2 * b3 - a3 * b2,
          a0 * b2 - a1 * b3 + a2 * b0 + a3 * b1,
          a0 * b3 + a1 * b2 - a2 * b1 + a3 * b0,
      ],
      dim=-1,
  )


def qdag(q):
  return q * q.new_tensor([1.0, -1.0, -1.0, -1.0])


def qnorm(q):
  return q / (torch.norm(q, dim=-1, keepdim=True).clamp(min=1e-14))


def get_staples(U, mu):
  S = torch.zeros_like(U[mu])
  for nu in range(4):
    if nu == mu:
      continue
    s1 = qmul(
        torch.roll(U[nu], shifts=-1, dims=mu),
        qmul(qdag(torch.roll(U[mu], shifts=-1, dims=nu)), qdag(U[nu])),
    )
    u_nu_bwd = torch.roll(U[nu], shifts=+1, dims=nu)
    # Исправлен сдвиг U[mu] на shifts=+1 для точного совпадения с координатой x - nu
    s2 = qmul(
        qdag(torch.roll(u_nu_bwd, shifts=-1, dims=mu)),
        qmul(qdag(torch.roll(U[mu], shifts=+1, dims=nu)), u_nu_bwd),
    )
    S = S + s1 + s2
  return S


def heatbath_sweep(U, beta, parity_mask):
  for mu in range(4):
    for p in (0, 1):
      mask = parity_mask == p
      S = get_staples(U, mu)
      S_m = S[mask]
      s_norm = torch.norm(S_m, dim=-1, keepdim=True).clamp(min=1e-12)
      V = S_m / s_norm
      alpha = (beta * s_norm).squeeze(-1)

      x = torch.zeros_like(alpha)
      accepted = torch.zeros(alpha.shape[0], dtype=torch.bool, device=device)
      for _ in range(12):
        r1 = torch.rand_like(alpha).clamp(min=1e-12)
        r2 = torch.rand_like(alpha).clamp(min=1e-12)
        r3 = torch.rand_like(alpha)
        r4 = torch.rand_like(alpha)
        cand_x = (
            -torch.log(r1) - torch.log(r2) * (torch.cos(2.0 * np.pi * r3) ** 2)
        ) / alpha
        ok = (cand_x <= 2.0) & (r4**2 <= (1.0 - 0.5 * cand_x))
        new_acc = (~accepted) & ok
        x = torch.where(new_acc, cand_x, x)
        accepted = accepted | ok

      x = torch.where(accepted, x, (0.5 / alpha).clamp(max=2.0))
      q0 = (1.0 - x).unsqueeze(-1)
      phi = 2.0 * np.pi * torch.rand_like(q0)
      ct = 2.0 * torch.rand_like(q0) - 1.0
      st = (1.0 - ct**2).clamp(min=0.0).sqrt()
      r = (1.0 - q0**2).clamp(min=0.0).sqrt()
      qv = torch.cat(
          [q0, r * st * torch.cos(phi), r * st * torch.sin(phi), r * ct], dim=-1
      )
      u_new = qmul(qv, qdag(V))
      U[mu, mask] = qnorm(u_new)
  return U


def overrelax_sweep(U, parity_mask):
  for mu in range(4):
    for p in (0, 1):
      mask = parity_mask == p
      S = get_staples(U, mu)
      S_m = S[mask]
      s_norm = torch.norm(S_m, dim=-1, keepdim=True).clamp(min=1e-12)
      V = S_m / s_norm
      V_dag = qdag(V)
      U_old = U[mu, mask]
      u_new = qmul(V_dag, qmul(qdag(U_old), V_dag))
      U[mu, mask] = qnorm(u_new)
  return U


def ape_step_spatial(U, alpha=0.20):
  U_new = U.clone()
  for mu in (1, 2, 3):
    C = torch.zeros_like(U[mu])
    for nu in (1, 2, 3):
      if nu == mu:
        continue
      u1 = U[nu]
      u2 = torch.roll(U[mu], shifts=-1, dims=nu)
      u3 = qdag(torch.roll(U[nu], shifts=-1, dims=mu))
      c_pos = qmul(u1, qmul(u2, u3))

      u_nu_bwd = torch.roll(U[nu], shifts=+1, dims=nu)
      u1_neg = qdag(u_nu_bwd)
      u2_neg = torch.roll(U[mu], shifts=+1, dims=nu)
      u3_neg = torch.roll(u_nu_bwd, shifts=-1, dims=mu)
      c_neg = qmul(u1_neg, qmul(u2_neg, u3_neg))

      C = C + c_pos + c_neg
    W = (1.0 - alpha) * U[mu] + (alpha / 4.0) * C
    U_new[mu] = qnorm(W)
  return U_new


def build_rect_loop(U_field, mu, nu, R=1, T=1):
  acc_mu = U_field[mu]
  for i in range(1, R):
    acc_mu = qmul(acc_mu, torch.roll(U_field[mu], shifts=-i, dims=mu))
  acc_nu = U_field[nu]
  for j in range(1, T):
    acc_nu = qmul(acc_nu, torch.roll(U_field[nu], shifts=-j, dims=nu))

  nu_shift_R = torch.roll(acc_nu, shifts=-R, dims=mu)
  mu_shift_T = torch.roll(acc_mu, shifts=-T, dims=nu)
  c12 = qmul(acc_mu, nu_shift_R)
  c34 = qmul(qdag(mu_shift_T), qdag(acc_nu))
  return qmul(c12, c34)


def measure_operators(U_sm):
  L = U_sm.shape[1]
  O_1x1 = torch.zeros((L, L, L, L), device=device)
  O_1x2 = torch.zeros((L, L, L, L), device=device)
  O_1x3 = torch.zeros((L, L, L, L), device=device)

  planes = [(1, 2), (1, 3), (2, 3)]
  for mu, nu in planes:
    P = build_rect_loop(U_sm, mu, nu, R=1, T=1)
    O_1x1 = O_1x1 + P[..., 0]

    W12 = build_rect_loop(U_sm, mu, nu, R=1, T=2)
    W21 = build_rect_loop(U_sm, nu, mu, R=1, T=2)
    O_1x2 = O_1x2 + W12[..., 0] + W21[..., 0]

    W13 = build_rect_loop(U_sm, mu, nu, R=1, T=3)
    W31 = build_rect_loop(U_sm, nu, mu, R=1, T=3)
    O_1x3 = O_1x3 + W13[..., 0] + W31[..., 0]

  return (
      (O_1x1 / 3.0).mean(dim=(1, 2, 3)),
      (O_1x2 / 6.0).mean(dim=(1, 2, 3)),
      (O_1x3 / 6.0).mean(dim=(1, 2, 3)),
  )


def measure_wilson_loops(
    U_hyb, R_values=(1, 2, 3, 4, 5), T_values=(1, 2, 3, 4, 5)
):
  def path_spatial(k, R):
    acc = U_hyb[k]
    for i in range(1, R):
      acc = qmul(acc, torch.roll(U_hyb[k], shifts=-i, dims=k))
    return acc

  def path_temporal(T):
    acc = U_hyb[0]
    for j in range(1, T):
      acc = qmul(acc, torch.roll(U_hyb[0], shifts=-j, dims=0))
    return acc

  results = {}
  for R in R_values:
    for T in T_values:
      val = 0.0
      for k in (1, 2, 3):
        Lk = path_spatial(k, R)
        T0 = path_temporal(T)
        T0_shift_R = torch.roll(T0, shifts=-R, dims=k)
        Lk_shift_T = torch.roll(Lk, shifts=-T, dims=0)
        c12 = qmul(Lk, T0_shift_R)
        c34 = qmul(qdag(Lk_shift_T), qdag(T0))
        loop = qmul(c12, c34)
        val += loop[..., 0].mean().item()
      results[(R, T)] = val / 3.0
  return results


def gevp_solve(C_mat, t0=1, t_target=2, rcond=1e-3):
  C0 = 0.5 * (C_mat[:, :, t0] + C_mat[:, :, t0].T)
  w0, v0 = np.linalg.eigh(C0)
  max_w = np.max(w0)
  if max_w <= 0:
    return np.nan, 0, np.inf
  good = w0 > (rcond * max_w)
  pos_w = w0[w0 > 0]
  cond = max_w / max(pos_w.min(), 1e-16) if len(pos_w) > 0 else np.inf
  if np.sum(good) < 2:
    return np.nan, int(np.sum(good)), cond

  inv_sqrt = v0[:, good] @ np.diag(1.0 / np.sqrt(w0[good]))
  Ct = 0.5 * (C_mat[:, :, t_target] + C_mat[:, :, t_target].T)
  C_tilde = inv_sqrt.T @ Ct @ inv_sqrt
  w_t = np.linalg.eigvalsh(C_tilde)
  w_pos = w_t[np.isfinite(w_t) & (w_t > 0)]
  if len(w_pos) == 0:
    return np.nan, int(np.sum(good)), cond
  lam = np.max(w_pos)
  return -np.log(lam) / (t_target - t0), int(np.sum(good)), cond


def build_C(G, O_mean):
  C = G - np.outer(O_mean, O_mean)[:, :, None]
  C = 0.5 * (C + C.transpose(1, 0, 2))
  L = C.shape[-1]
  Cs = np.zeros_like(C)
  Cs[:, :, 0] = C[:, :, 0]
  for dt in range(1, L):
    Cs[:, :, dt] = 0.5 * (C[:, :, dt] + C[:, :, L - dt])
  return Cs


def cornell(R, V0, sigma, e):
  return V0 + sigma * R - e / R


def safe_fit_cornell(V_arr, R_arr):
  mask = (R_arr >= 2) & np.isfinite(V_arr)
  if np.sum(mask) < 3:
    return np.array([np.nan, np.nan, np.nan])
  try:
    popt, _ = curve_fit(
        cornell,
        R_arr[mask],
        V_arr[mask],
        p0=[0.60, 0.135, 0.26],
        bounds=([-2.0, 0.01, 0.0], [3.0, 0.40, 1.0]),
        maxfev=15000,
    )
    return popt
  except Exception:
    return np.array([np.nan, np.nan, np.nan])


def calc_jk_err(arr):
  v = arr[np.isfinite(arr)]
  if len(v) < 2:
    return np.nan
  return np.sqrt((len(v) - 1) / len(v) * np.sum((v - v.mean()) ** 2))


def safe_log_ratio(a, b):
  if np.isfinite(a) and np.isfinite(b) and a > 0 and b > 0:
    return np.log(a / b)
  return np.nan


def get_sha256(path):
  sha = hashlib.sha256()
  with open(path, 'rb') as f:
    while chunk := f.read(8192 * 1024):
      sha.update(chunk)
  return sha.hexdigest()


def run_point_beta23(
    L=12,
    beta=2.3,
    n_therm=100,
    n_meas=2000,
    n_overrelax=4,
    n_bins=40,
    ape_levels=(2, 6, 12),
    ape_alpha=0.20,
    loop_smear=6,
    R_values=(1, 2, 3, 4, 5),
    T_values=(1, 2, 3, 4, 5),
):

  print('=' * 75)
  print(f'SU(2) GLUEBALL SPECTROSCOPY & STRING TENSION: beta={beta}, L={L}^4')
  print(
      f'Statistics: {n_meas} configs, {n_bins} bins ({n_meas//n_bins} cfg/bin),'
      f' OR/HB: {n_overrelax}'
  )
  print(f'APE Scheme: {ape_levels}, alpha={ape_alpha} | loop_smear={loop_smear}')
  print('=' * 75)

  U = torch.zeros((4, L, L, L, L, 4), device=device)
  U[..., 0] = 1.0
  coords = torch.stack(
      torch.meshgrid([torch.arange(L, device=device)] * 4, indexing='ij')
  )
  parity = (coords.sum(0)) % 2

  def mean_plaquette(Ux):
    tot, cnt = 0.0, 0
    for mu in range(4):
      for nu in range(mu + 1, 4):
        u_nu_fwd = torch.roll(Ux[nu], shifts=-1, dims=mu)
        u_mu_fwd = torch.roll(Ux[mu], shifts=-1, dims=nu)
        P = qmul(
            Ux[mu], qmul(u_nu_fwd, qmul(qdag(u_mu_fwd), qdag(Ux[nu])))
        )
        tot += P[..., 0].mean().item()
        cnt += 1
    return tot / cnt

  print(f'[Sanity 1] Cold Start Plaquette: {mean_plaquette(U):.6f}')

  print('\nStage 1: Thermalization...')
  t_start = time.time()
  for step in range(n_therm):
    U = heatbath_sweep(U, beta, parity)
    for _ in range(n_overrelax):
      U = overrelax_sweep(U, parity)
    if (step + 1) % 25 == 0:
      print(
          f'  Therm {step+1:3d}/{n_therm} | <P> = {mean_plaquette(U):.6f} |'
          f' {time.time()-t_start:.1f}s'
      )

  n_lev = len(ape_levels)
  n_ops = 3 * n_lev
  O_data = np.zeros((n_meas, n_ops, L), dtype=np.float64)
  plaq_arr = np.zeros(n_meas, dtype=np.float64)
  loop_keys = [(R, T) for R in R_values for T in T_values]
  W_data = np.zeros((n_meas, len(loop_keys)), dtype=np.float64)

  print(f'\nStage 2: Production ({n_meas} trajectories)...')
  t_start = time.time()
  for m in range(n_meas):
    U = heatbath_sweep(U, beta, parity)
    for _ in range(n_overrelax):
      U = overrelax_sweep(U, parity)
    plaq_arr[m] = mean_plaquette(U)

    U_sm = U.clone()
    cur = 0
    for li, target in enumerate(ape_levels):
      while cur < target:
        U_sm = ape_step_spatial(U_sm, ape_alpha)
        cur += 1
      o1, o2, o3 = measure_operators(U_sm)
      O_data[m, 3 * li + 0, :] = o1.cpu().numpy()
      O_data[m, 3 * li + 1, :] = o2.cpu().numpy()
      O_data[m, 3 * li + 2, :] = o3.cpu().numpy()

      if cur == loop_smear:
        U_hyb = U_sm.clone()
        U_hyb[0] = U[0]
        w_dict = measure_wilson_loops(U_hyb, R_values, T_values)
        for ki, k in enumerate(loop_keys):
          W_data[m, ki] = w_dict[k]

    if (m + 1) % 200 == 0:
      print(
          f'  Config {m+1:4d}/{n_meas} | Running <P> ='
          f' {plaq_arr[:m+1].mean():.6f} | {time.time()-t_start:.1f}s'
      )

  fn_data = 'su2_production_beta23_L12.npz'
  np.savez(fn_data, O_data=O_data, W_data=W_data, plaq_arr=plaq_arr)
  print(f"  [Auto-Save] Конфигурации сохранены в '{fn_data}'")

  bin_size = n_meas // n_bins
  n_use = n_bins * bin_size

  G_cfg = np.zeros((n_meas, n_ops, n_ops, L), dtype=np.float64)
  for dt in range(L):
    G_cfg[:, :, :, dt] = (
        np.einsum('mit,mjt->mij', O_data, np.roll(O_data, -dt, axis=-1)) / L
    )
  O_mean_cfg = O_data.mean(axis=-1)

  G_binned = (
      G_cfg[:n_use].reshape(n_bins, bin_size, n_ops, n_ops, L).mean(axis=1)
  )
  O_binned = O_mean_cfg[:n_use].reshape(n_bins, bin_size, n_ops).mean(axis=1)
  W_binned = (
      W_data[:n_use].reshape(n_bins, bin_size, len(loop_keys)).mean(axis=1)
  )

  C_full = build_C(G_binned.mean(0), O_binned.mean(0))
  C_jk_all = np.zeros((n_bins, n_ops, n_ops, L), dtype=np.float64)
  for b in range(n_bins):
    Gb = np.delete(G_binned, b, axis=0).mean(axis=0)
    Ob = np.delete(O_binned, b, axis=0).mean(axis=0)
    C_jk_all[b] = build_C(Gb, Ob)

  W_mean_full = {k: W_binned.mean(0)[ki] for ki, k in enumerate(loop_keys)}
  R_arr = np.array(R_values)
  V_T3 = np.array([
      safe_log_ratio(W_mean_full[(R, 3)], W_mean_full[(R, 4)]) for R in R_values
  ])
  popt = safe_fit_cornell(V_T3, R_arr)
  sigma_a2 = popt[1]

  sigma_jk = np.full(n_bins, np.nan)
  for b in range(n_bins):
    W_sub = np.delete(W_binned, b, axis=0).mean(axis=0)
    W_map = {k: W_sub[ki] for ki, k in enumerate(loop_keys)}
    Vb = np.array(
        [safe_log_ratio(W_map[(R, 3)], W_map[(R, 4)]) for R in R_values]
    )
    popt_b = safe_fit_cornell(Vb, R_arr)
    sigma_jk[b] = popt_b[1]
  sigma_err = calc_jk_err(sigma_jk)

  b_hybrid = [0, 4, 8]
  m_hyb_12, ng_h12, cond_h = gevp_solve(
      C_full[np.ix_(b_hybrid, b_hybrid)], t0=1, t_target=2
  )
  m_hyb_23, ng_h23, _ = gevp_solve(
      C_full[np.ix_(b_hybrid, b_hybrid)], t0=2, t_target=3
  )

  m_hyb_jk = np.full(n_bins, np.nan)
  for b in range(n_bins):
    m_hyb_jk[b], _, _ = gevp_solve(
        C_jk_all[b][np.ix_(b_hybrid, b_hybrid)], t0=1, t_target=2
    )
  m_hyb_err = calc_jk_err(m_hyb_jk)

  b_smear = [0, 3, 6]
  m_sm_12, ng_sm, cond_sm = gevp_solve(
      C_full[np.ix_(b_smear, b_smear)], t0=1, t_target=2
  )

  b_geom = [6, 7, 8]
  m_gm_12, ng_gm, cond_gm = gevp_solve(
      C_full[np.ix_(b_geom, b_geom)], t0=1, t_target=2
  )

  R0_jk = np.array([
      m_hyb_jk[b] / np.sqrt(sigma_jk[b])
      for b in range(n_bins)
      if np.isfinite(m_hyb_jk[b])
      and np.isfinite(sigma_jk[b])
      and m_hyb_jk[b] > 0
      and sigma_jk[b] > 0
  ])
  R_0 = (
      m_hyb_12 / np.sqrt(sigma_a2)
      if (
          np.isfinite(m_hyb_12)
          and np.isfinite(sigma_a2)
          and m_hyb_12 > 0
          and sigma_a2 > 0
      )
      else np.nan
  )
  R_0_err = calc_jk_err(R0_jk)

  data_hash = get_sha256(fn_data)
  size_mb = os.path.getsize(fn_data) / (1024 * 1024)
  fn_manifest = 'manifest_beta23_L12.json'

  manifest = {
      'experiment': 'SU(2) Glueball & String Tension Scaling Point',
      'timestamp_utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime()),
      'lattice': {
          'group': 'SU(2)',
          'beta': float(beta),
          'geometry': f'{L}^4',
          'action': 'Wilson',
      },
      'monte_carlo': {
          'n_therm': int(n_therm),
          'n_meas': int(n_meas),
          'n_overrelax': int(n_overrelax),
          'n_bins': int(n_bins),
      },
      'gevp_diagnostics': {
          'primary_hybrid': {
              'basis': ['1x1_A2', '1x2_A6', '1x3_A12'],
              'cond': round(float(cond_h), 1),
              'modes': f'{ng_h12}/3',
              'm_t1_t2': round(float(m_hyb_12), 4),
              'm_t2_t3': round(float(m_hyb_23), 4),
          },
          'control_smearing': {
              'basis': ['1x1_A2', '1x1_A6', '1x1_A12'],
              'cond': round(float(cond_sm), 1),
              'modes': f'{ng_sm}/3',
              'm_t1_t2': round(float(m_sm_12), 4),
          },
          'control_geometry': {
              'basis': ['1x1_A12', '1x2_A12', '1x3_A12'],
              'cond': round(float(cond_gm), 1),
              'modes': f'{ng_gm}/3',
              'm_t1_t2': round(float(m_gm_12), 4),
          },
      },
      'physical_results': {
          'plaquette': round(float(plaq_arr.mean()), 6),
          'plaquette_err': (
              round(float(plaq_arr.std(ddof=1) / np.sqrt(n_meas)), 6)
          ),
          'm_0pp_a': (
              round(float(m_hyb_12), 4) if np.isfinite(m_hyb_12) else None
          ),
          'm_0pp_a_err': (
              round(float(m_hyb_err), 4) if np.isfinite(m_hyb_err) else None
          ),
          'sigma_a2': (
              round(float(sigma_a2), 5) if np.isfinite(sigma_a2) else None
          ),
          'sigma_a2_err': (
              round(float(sigma_err), 5) if np.isfinite(sigma_err) else None
          ),
          'coulomb_e': (
              round(float(popt[2]), 4) if np.isfinite(popt[2]) else None
          ),
          'mass_ratio_R0': round(float(R_0), 3) if np.isfinite(R_0) else None,
          'mass_ratio_R0_err': (
              round(float(R_0_err), 3) if np.isfinite(R_0_err) else None
          ),
      },
      'artifact': {
          'file': fn_data,
          'size_mb': round(size_mb, 2),
          'sha256': data_hash,
      },
  }

  with open(fn_manifest, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=4, ensure_ascii=False)
  manifest_hash = get_sha256(fn_manifest)

  print('\n' + '=' * 75)
  print(f'ИТОГОВЫЕ ФИЗИЧЕСКИЕ РЕЗУЛЬТАТЫ (BETA = {beta}, L = {L})')
  print('=' * 75)
  print(
      f'  <P> (Wilson Plaquette):   {plaq_arr.mean():.6f} +/='
      f' {plaq_arr.std(ddof=1)/np.sqrt(n_meas):.6f}'
  )
  print(f'  String Tension a^2*sigma: {sigma_a2:.5f} +/- {sigma_err:.5f}')
  print(
      f'  Coulomb parameter e:      {popt[2]:.4f} (Lüscher ref: pi/12 = 0.2618)'
  )
  print('-' * 75)
  print('  GEVP Stability Spectrum:')
  print(
      '    Primary Hybrid   [0,4,8] (t0=1->t=2): m*a ='
      f' {m_hyb_12:.4f} +/- {m_hyb_err:.4f} | cond={cond_h:.1f} ({ng_h12}/3'
      ' modes)'
  )
  print(
      '    Primary Hybrid   [0,4,8] (t0=2->t=3): m*a ='
      f' {m_hyb_23:.4f} (noise/plateau audit)'
  )
  print(
      '    Pure Smearing    [0,3,6] (t0=1->t=2): m*a ='
      f' {m_sm_12:.4f} | cond={cond_sm:.1f} ({ng_sm}/3 modes)'
  )
  print(
      '    Pure Geometry    [6,7,8] (t0=1->t=2): m*a ='
      f' {m_gm_12:.4f} | cond={cond_gm:.1f} ({ng_gm}/3 modes)'
  )
  print('-' * 75)
  print(f'  MASS RATIO R0 = m / √σ:   {R_0:.3f} +/- {R_0_err:.3f}')
  print('  Expected Physical Range:  3.40 - 3.75')
  print('=' * 75)
  print(f'  DATA SHA-256:     {data_hash}')
  print(f'  MANIFEST SHA-256: {manifest_hash}')
  print('=' * 75)


if __name__ == '__main__':
  run_point_beta23()

In [ ]:
from IPython.display import FileLink, display
display(FileLink('su2_production_beta23_L12.npz'))
display(FileLink('manifest_beta23_L12.json'))

In [ ]:
# ==============================================================================
# SU(2) Glueball GEVP + Cornell Potential — Continuum Point beta=2.5, L=16^4 (v2.6-beta25-L16)
# Baselines: Hybrid [0,4,8], Pure-Smear [0,3,6], Pure-Geometry [6,7,8]
# Full Jackknife, Protected Cornell Fit, Automated SHA-256 Manifest
# ==============================================================================
import torch
import numpy as np
from scipy.optimize import curve_fit
import hashlib, json, time, os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active Compute Device: {device}")

# 1. Алгебра кватернионов SU(2)
def qmul(a, b):
    a0, a1, a2, a3 = a.unbind(-1)
    b0, b1, b2, b3 = b.unbind(-1)
    return torch.stack([
        a0*b0 - a1*b1 - a2*b2 - a3*b3,
        a0*b1 + a1*b0 + a2*b3 - a3*b2,
        a0*b2 - a1*b3 + a2*b0 + a3*b1,
        a0*b3 + a1*b2 - a2*b1 + a3*b0
    ], dim=-1)

def qdag(q):
    return q * q.new_tensor([1., -1., -1., -1.])

def qnorm(q):
    return q / (torch.norm(q, dim=-1, keepdim=True).clamp(min=1e-14))

# 2. Стапели с выверенной геометрией контура
def get_staples(U, mu):
    S = torch.zeros_like(U[mu])
    for nu in range(4):
        if nu == mu: continue
        s1 = qmul(
            torch.roll(U[nu], shifts=-1, dims=mu),
            qmul(qdag(torch.roll(U[mu], shifts=-1, dims=nu)), qdag(U[nu]))
        )
        u_nu_bwd = torch.roll(U[nu], shifts=+1, dims=nu)
        s2 = qmul(
            qdag(torch.roll(u_nu_bwd, shifts=-1, dims=mu)),
            qmul(qdag(torch.roll(U[mu], shifts=+1, dims=nu)), u_nu_bwd)
        )
        S = S + s1 + s2
    return S

# 3. Heatbath & Overrelaxation
def heatbath_sweep(U, beta, parity_mask):
    for mu in range(4):
        for p in (0, 1):
            mask = (parity_mask == p)
            S = get_staples(U, mu)
            S_m = S[mask]
            s_norm = torch.norm(S_m, dim=-1, keepdim=True).clamp(min=1e-12)
            V = S_m / s_norm
            alpha = (beta * s_norm).squeeze(-1)

            x = torch.zeros_like(alpha)
            accepted = torch.zeros(alpha.shape[0], dtype=torch.bool, device=device)
            for _ in range(12):
                r1 = torch.rand_like(alpha).clamp(min=1e-12)
                r2 = torch.rand_like(alpha).clamp(min=1e-12)
                r3 = torch.rand_like(alpha)
                r4 = torch.rand_like(alpha)
                cand_x = (-torch.log(r1) - torch.log(r2) * (torch.cos(2.0 * np.pi * r3)**2)) / alpha
                ok = (cand_x <= 2.0) & (r4**2 <= (1.0 - 0.5 * cand_x))
                new_acc = (~accepted) & ok
                x = torch.where(new_acc, cand_x, x)
                accepted = accepted | ok

            x = torch.where(accepted, x, (0.5 / alpha).clamp(max=2.0))
            q0 = (1.0 - x).unsqueeze(-1)
            phi = 2.0 * np.pi * torch.rand_like(q0)
            ct = 2.0 * torch.rand_like(q0) - 1.0
            st = (1.0 - ct**2).clamp(min=0.0).sqrt()
            r = (1.0 - q0**2).clamp(min=0.0).sqrt()
            qv = torch.cat([q0, r*st*torch.cos(phi), r*st*torch.sin(phi), r*ct], dim=-1)
            u_new = qmul(qv, qdag(V))
            U[mu, mask] = qnorm(u_new)
    return U

def overrelax_sweep(U, parity_mask):
    for mu in range(4):
        for p in (0, 1):
            mask = (parity_mask == p)
            S = get_staples(U, mu)
            S_m = S[mask]
            s_norm = torch.norm(S_m, dim=-1, keepdim=True).clamp(min=1e-12)
            V = S_m / s_norm
            V_dag = qdag(V)
            U_old = U[mu, mask]
            u_new = qmul(V_dag, qmul(qdag(U_old), V_dag))
            U[mu, mask] = qnorm(u_new)
    return U

# 4. Пространственное APE-сглаживание
def ape_step_spatial(U, alpha=0.20):
    U_new = U.clone()
    for mu in (1, 2, 3):
        C = torch.zeros_like(U[mu])
        for nu in (1, 2, 3):
            if nu == mu: continue
            u1 = U[nu]
            u2 = torch.roll(U[mu], shifts=-1, dims=nu)
            u3 = qdag(torch.roll(U[nu], shifts=-1, dims=mu))
            c_pos = qmul(u1, qmul(u2, u3))

            u_nu_bwd = torch.roll(U[nu], shifts=+1, dims=nu)
            u1_neg = qdag(u_nu_bwd)
            u2_neg = torch.roll(U[mu], shifts=+1, dims=nu)
            u3_neg = torch.roll(u_nu_bwd, shifts=-1, dims=mu)
            c_neg = qmul(u1_neg, qmul(u2_neg, u3_neg))

            C = C + c_pos + c_neg
        W = (1.0 - alpha) * U[mu] + (alpha / 4.0) * C
        U_new[mu] = qnorm(W)
    return U_new

# 5. Операторы и петли Вильсона
def build_rect_loop(U_field, mu, nu, R=1, T=1):
    acc_mu = U_field[mu]
    for i in range(1, R):
        acc_mu = qmul(acc_mu, torch.roll(U_field[mu], shifts=-i, dims=mu))
    acc_nu = U_field[nu]
    for j in range(1, T):
        acc_nu = qmul(acc_nu, torch.roll(U_field[nu], shifts=-j, dims=nu))

    nu_shift_R = torch.roll(acc_nu, shifts=-R, dims=mu)
    mu_shift_T = torch.roll(acc_mu, shifts=-T, dims=nu)
    c12 = qmul(acc_mu, nu_shift_R)
    c34 = qmul(qdag(mu_shift_T), qdag(acc_nu))
    return qmul(c12, c34)

def measure_operators(U_sm):
    L = U_sm.shape[1]
    O_1x1 = torch.zeros((L, L, L, L), device=device)
    O_1x2 = torch.zeros((L, L, L, L), device=device)
    O_1x3 = torch.zeros((L, L, L, L), device=device)

    planes = [(1, 2), (1, 3), (2, 3)]
    for mu, nu in planes:
        P = build_rect_loop(U_sm, mu, nu, R=1, T=1)
        O_1x1 = O_1x1 + P[..., 0]

        W12 = build_rect_loop(U_sm, mu, nu, R=1, T=2)
        W21 = build_rect_loop(U_sm, nu, mu, R=1, T=2)
        O_1x2 = O_1x2 + W12[..., 0] + W21[..., 0]

        W13 = build_rect_loop(U_sm, mu, nu, R=1, T=3)
        W31 = build_rect_loop(U_sm, nu, mu, R=1, T=3)
        O_1x3 = O_1x3 + W13[..., 0] + W31[..., 0]

    return (O_1x1 / 3.0).mean(dim=(1, 2, 3)), (O_1x2 / 6.0).mean(dim=(1, 2, 3)), (O_1x3 / 6.0).mean(dim=(1, 2, 3))

def measure_wilson_loops(U_hyb, R_values=(1, 2, 3, 4, 5), T_values=(1, 2, 3, 4, 5)):
    def path_spatial(k, R):
        acc = U_hyb[k]
        for i in range(1, R):
            acc = qmul(acc, torch.roll(U_hyb[k], shifts=-i, dims=k))
        return acc

    def path_temporal(T):
        acc = U_hyb[0]
        for j in range(1, T):
            acc = qmul(acc, torch.roll(U_hyb[0], shifts=-j, dims=0))
        return acc

    results = {}
    for R in R_values:
        for T in T_values:
            val = 0.0
            for k in (1, 2, 3):
                Lk = path_spatial(k, R)
                T0 = path_temporal(T)
                T0_shift_R = torch.roll(T0, shifts=-R, dims=k)
                Lk_shift_T = torch.roll(Lk, shifts=-T, dims=0)
                c12 = qmul(Lk, T0_shift_R)
                c34 = qmul(qdag(Lk_shift_T), qdag(T0))
                loop = qmul(c12, c34)
                val += loop[..., 0].mean().item()
            results[(R, T)] = val / 3.0
    return results

# 6. GEVP и численный анализ
def gevp_solve(C_mat, t0=1, t_target=2, rcond=1e-3):
    C0 = 0.5 * (C_mat[:, :, t0] + C_mat[:, :, t0].T)
    w0, v0 = np.linalg.eigh(C0)
    max_w = np.max(w0)
    if max_w <= 0: return np.nan, 0, np.inf
    good = w0 > (rcond * max_w)
    pos_w = w0[w0 > 0]
    cond = max_w / max(pos_w.min(), 1e-16) if len(pos_w) > 0 else np.inf
    if np.sum(good) < 2: return np.nan, int(np.sum(good)), cond

    inv_sqrt = v0[:, good] @ np.diag(1.0 / np.sqrt(w0[good]))
    Ct = 0.5 * (C_mat[:, :, t_target] + C_mat[:, :, t_target].T)
    C_tilde = inv_sqrt.T @ Ct @ inv_sqrt
    w_t = np.linalg.eigvalsh(C_tilde)
    w_pos = w_t[np.isfinite(w_t) & (w_t > 0)]
    if len(w_pos) == 0: return np.nan, int(np.sum(good)), cond
    lam = np.max(w_pos)
    return -np.log(lam) / (t_target - t0), int(np.sum(good)), cond

def build_C(G, O_mean):
    C = G - np.outer(O_mean, O_mean)[:, :, None]
    C = 0.5 * (C + C.transpose(1, 0, 2))
    L = C.shape[-1]
    Cs = np.zeros_like(C)
    Cs[:, :, 0] = C[:, :, 0]
    for dt in range(1, L):
        Cs[:, :, dt] = 0.5 * (C[:, :, dt] + C[:, :, L - dt])
    return Cs

def cornell(R, V0, sigma, e):
    return V0 + sigma * R - e / R

# Границы адаптированы под бета=2.5 (sigma ~ 0.045 - 0.055)
def safe_fit_cornell(V_arr, R_arr):
    mask = (R_arr >= 2) & np.isfinite(V_arr)
    if np.sum(mask) < 3: return np.array([np.nan, np.nan, np.nan])
    try:
        popt, _ = curve_fit(
            cornell, R_arr[mask], V_arr[mask],
            p0=[0.50, 0.048, 0.26],
            bounds=([-2.0, 0.005, 0.0], [3.0, 0.25, 1.0]),
            maxfev=15000
        )
        return popt
    except Exception:
        return np.array([np.nan, np.nan, np.nan])

def calc_jk_err(arr):
    v = arr[np.isfinite(arr)]
    if len(v) < 2: return np.nan
    return np.sqrt((len(v) - 1) / len(v) * np.sum((v - v.mean())**2))

def safe_log_ratio(a, b):
    if np.isfinite(a) and np.isfinite(b) and a > 0 and b > 0:
        return np.log(a / b)
    return np.nan

def get_sha256(path):
    sha = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(8192 * 1024):
            sha.update(chunk)
    return sha.hexdigest()

# 7. Конвейер симуляции и анализа
def run_point_beta25_L16(L=16, beta=2.5, n_therm=100, n_meas=2000,
                         n_overrelax=4, n_bins=40,
                         ape_levels=(2, 6, 12), ape_alpha=0.20, loop_smear=6,
                         R_values=(1, 2, 3, 4, 5), T_values=(1, 2, 3, 4, 5)):

    print("="*75)
    print(f"SU(2) GLUEBALL SPECTROSCOPY & STRING TENSION: beta={beta}, L={L}^4")
    print(f"Statistics: {n_meas} configs, {n_bins} bins ({n_meas//n_bins} cfg/bin), OR/HB: {n_overrelax}")
    print(f"APE Scheme: {ape_levels}, alpha={ape_alpha} | loop_smear={loop_smear}")
    print("="*75)

    U = torch.zeros((4, L, L, L, L, 4), device=device)
    U[..., 0] = 1.0
    coords = torch.stack(torch.meshgrid([torch.arange(L, device=device)]*4, indexing='ij'))
    parity = (coords.sum(0)) % 2

    def mean_plaquette(Ux):
        tot, cnt = 0.0, 0
        for mu in range(4):
            for nu in range(mu+1, 4):
                u_nu_fwd = torch.roll(Ux[nu], shifts=-1, dims=mu)
                u_mu_fwd = torch.roll(Ux[mu], shifts=-1, dims=nu)
                P = qmul(Ux[mu], qmul(u_nu_fwd, qmul(qdag(u_mu_fwd), qdag(Ux[nu]))))
                tot += P[..., 0].mean().item()
                cnt += 1
        return tot / cnt

    print(f"[Sanity 1] Cold Start Plaquette: {mean_plaquette(U):.6f}")

    print("\nStage 1: Thermalization...")
    t_start = time.time()
    for step in range(n_therm):
        U = heatbath_sweep(U, beta, parity)
        for _ in range(n_overrelax):
            U = overrelax_sweep(U, parity)
        if (step + 1) % 25 == 0:
            print(f"  Therm {step+1:3d}/{n_therm} | <P> = {mean_plaquette(U):.6f} | {time.time()-t_start:.1f}s")

    n_lev = len(ape_levels)
    n_ops = 3 * n_lev
    op_names = {
        0: '1x1_A2',  1: '1x2_A2',  2: '1x3_A2',
        3: '1x1_A6',  4: '1x2_A6',  5: '1x3_A6',
        6: '1x1_A12', 7: '1x2_A12', 8: '1x3_A12'
    }

    O_data = np.zeros((n_meas, n_ops, L), dtype=np.float64)
    plaq_arr = np.zeros(n_meas, dtype=np.float64)
    loop_keys = [(R, T) for R in R_values for T in T_values]
    W_data = np.zeros((n_meas, len(loop_keys)), dtype=np.float64)

    print(f"\nStage 2: Production ({n_meas} trajectories)...")
    t_start = time.time()
    for m in range(n_meas):
        U = heatbath_sweep(U, beta, parity)
        for _ in range(n_overrelax):
            U = overrelax_sweep(U, parity)
        plaq_arr[m] = mean_plaquette(U)

        U_sm = U.clone()
        cur = 0
        for li, target in enumerate(ape_levels):
            while cur < target:
                U_sm = ape_step_spatial(U_sm, ape_alpha)
                cur += 1
            o1, o2, o3 = measure_operators(U_sm)
            O_data[m, 3*li + 0, :] = o1.cpu().numpy()
            O_data[m, 3*li + 1, :] = o2.cpu().numpy()
            O_data[m, 3*li + 2, :] = o3.cpu().numpy()

            if cur == loop_smear:
                U_hyb = U_sm.clone()
                U_hyb[0] = U[0]
                w_dict = measure_wilson_loops(U_hyb, R_values, T_values)
                for ki, k in enumerate(loop_keys):
                    W_data[m, ki] = w_dict[k]

        if (m + 1) % 100 == 0:
            print(f"  Config {m+1:4d}/{n_meas} | Running <P> = {plaq_arr[:m+1].mean():.6f} | {time.time()-t_start:.1f}s")

    fn_data = 'su2_production_beta25_L16.npz'
    np.savez(fn_data, O_data=O_data, W_data=W_data, plaq_arr=plaq_arr)
    print(f"  [Auto-Save] Конфигурации сохранены в '{fn_data}'")

    # Формирование бинов
    bin_size = n_meas // n_bins
    n_use = n_bins * bin_size

    G_cfg = np.zeros((n_meas, n_ops, n_ops, L), dtype=np.float64)
    for dt in range(L):
        G_cfg[:, :, :, dt] = np.einsum('mit,mjt->mij', O_data, np.roll(O_data, -dt, axis=-1)) / L
    O_mean_cfg = O_data.mean(axis=-1)

    G_binned = G_cfg[:n_use].reshape(n_bins, bin_size, n_ops, n_ops, L).mean(axis=1)
    O_binned = O_mean_cfg[:n_use].reshape(n_bins, bin_size, n_ops).mean(axis=1)
    W_binned = W_data[:n_use].reshape(n_bins, bin_size, len(loop_keys)).mean(axis=1)

    C_full = build_C(G_binned.mean(0), O_binned.mean(0))
    C_jk_all = np.zeros((n_bins, n_ops, n_ops, L), dtype=np.float64)
    for b in range(n_bins):
        Gb = np.delete(G_binned, b, axis=0).mean(axis=0)
        Ob = np.delete(O_binned, b, axis=0).mean(axis=0)
        C_jk_all[b] = build_C(Gb, Ob)

    # Потенциал Корнелла
    W_mean_full = {k: W_binned.mean(0)[ki] for ki, k in enumerate(loop_keys)}
    R_arr = np.array(R_values)
    V_T3 = np.array([safe_log_ratio(W_mean_full[(R, 3)], W_mean_full[(R, 4)]) for R in R_values])
    popt = safe_fit_cornell(V_T3, R_arr)
    sigma_a2 = popt[1]

    sigma_jk = np.full(n_bins, np.nan)
    for b in range(n_bins):
        W_sub = np.delete(W_binned, b, axis=0).mean(axis=0)
        W_map = {k: W_sub[ki] for ki, k in enumerate(loop_keys)}
        Vb = np.array([safe_log_ratio(W_map[(R, 3)], W_map[(R, 4)]) for R in R_values])
        popt_b = safe_fit_cornell(Vb, R_arr)
        sigma_jk[b] = popt_b[1]
    sigma_err = calc_jk_err(sigma_jk)

    # GEVP Базисы
    # 1. Основной: Гибридный
    b_hybrid = [0, 4, 8]  # {1x1_A2, 1x2_A6, 1x3_A12}
    m_hyb_12, ng_h12, cond_h = gevp_solve(C_full[np.ix_(b_hybrid, b_hybrid)], t0=1, t_target=2)
    m_hyb_23, ng_h23, _      = gevp_solve(C_full[np.ix_(b_hybrid, b_hybrid)], t0=2, t_target=3)

    m_hyb_jk = np.full(n_bins, np.nan)
    for b in range(n_bins):
        m_hyb_jk[b], _, _ = gevp_solve(C_jk_all[b][np.ix_(b_hybrid, b_hybrid)], t0=1, t_target=2)
    m_hyb_err = calc_jk_err(m_hyb_jk)

    # 2. Контрольный: Чистое сглаживание 1x1
    b_smear = [0, 3, 6]   # {1x1_A2, 1x1_A6, 1x1_A12}
    m_sm_12, ng_sm, cond_sm = gevp_solve(C_full[np.ix_(b_smear, b_smear)], t0=1, t_target=2)

    # 3. Контрольный: Чистая геометрия APE=12
    b_geom = [6, 7, 8]    # {1x1_A12, 1x2_A12, 1x3_A12}
    m_gm_12, ng_gm, cond_gm = gevp_solve(C_full[np.ix_(b_geom, b_geom)], t0=1, t_target=2)

    # Инвариант R_0
    R0_jk = np.array([
        m_hyb_jk[b] / np.sqrt(sigma_jk[b])
        for b in range(n_bins)
        if np.isfinite(m_hyb_jk[b]) and np.isfinite(sigma_jk[b]) and m_hyb_jk[b] > 0 and sigma_jk[b] > 0
    ])
    R_0 = m_hyb_12 / np.sqrt(sigma_a2) if (np.isfinite(m_hyb_12) and np.isfinite(sigma_a2) and m_hyb_12 > 0 and sigma_a2 > 0) else np.nan
    R_0_err = calc_jk_err(R0_jk)

    # Фиксация в манифест
    data_hash = get_sha256(fn_data)
    size_mb = os.path.getsize(fn_data) / (1024 * 1024)
    fn_manifest = 'manifest_beta25_L16.json'

    manifest = {
        "experiment": "SU(2) Glueball & String Tension Continuum Scaling Point",
        "timestamp_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
        "lattice": {"group": "SU(2)", "beta": float(beta), "geometry": f"{L}^4", "action": "Wilson"},
        "monte_carlo": {"n_therm": int(n_therm), "n_meas": int(n_meas), "n_overrelax": int(n_overrelax), "n_bins": int(n_bins)},
        "gevp_diagnostics": {
            "primary_hybrid": {"basis": ["1x1_A2", "1x2_A6", "1x3_A12"], "cond": round(float(cond_h), 1), "modes": f"{ng_h12}/3", "m_t1_t2": round(float(m_hyb_12), 4) if np.isfinite(m_hyb_12) else None, "m_t2_t3": round(float(m_hyb_23), 4) if np.isfinite(m_hyb_23) else None},
            "control_smearing": {"basis": ["1x1_A2", "1x1_A6", "1x1_A12"], "cond": round(float(cond_sm), 1), "modes": f"{ng_sm}/3", "m_t1_t2": round(float(m_sm_12), 4) if np.isfinite(m_sm_12) else None},
            "control_geometry": {"basis": ["1x1_A12", "1x2_A12", "1x3_A12"], "cond": round(float(cond_gm), 1), "modes": f"{ng_gm}/3", "m_t1_t2": round(float(m_gm_12), 4) if np.isfinite(m_gm_12) else None}
        },
        "physical_results": {
            "plaquette": round(float(plaq_arr.mean()), 6),
            "plaquette_err": round(float(plaq_arr.std(ddof=1) / np.sqrt(n_meas)), 6),
            "m_0pp_a": round(float(m_hyb_12), 4) if np.isfinite(m_hyb_12) else None,
            "m_0pp_a_err": round(float(m_hyb_err), 4) if np.isfinite(m_hyb_err) else None,
            "sigma_a2": round(float(sigma_a2), 5) if np.isfinite(sigma_a2) else None,
            "sigma_a2_err": round(float(sigma_err), 5) if np.isfinite(sigma_err) else None,
            "coulomb_e": round(float(popt[2]), 4) if np.isfinite(popt[2]) else None,
            "mass_ratio_R0": round(float(R_0), 3) if np.isfinite(R_0) else None,
            "mass_ratio_R0_err": round(float(R_0_err), 3) if np.isfinite(R_0_err) else None
        },
        "artifact": {"file": fn_data, "size_mb": round(size_mb, 2), "sha256": data_hash}
    }

    with open(fn_manifest, 'w', encoding='utf-8') as f:
        json.dump(manifest, f, indent=4, ensure_ascii=False)
    manifest_hash = get_sha256(fn_manifest)

    # Итоговый вывод
    print("\n" + "="*75)
    print(f"ИТОГОВЫЕ ФИЗИЧЕСКИЕ РЕЗУЛЬТАТЫ (BETA = {beta}, L = {L})")
    print("="*75)
    print(f"  <P> (Wilson Plaquette):   {plaq_arr.mean():.6f} +/- {plaq_arr.std(ddof=1)/np.sqrt(n_meas):.6f}")
    print(f"  String Tension a^2*sigma: {sigma_a2:.5f} +/- {sigma_err:.5f}")
    print(f"  Coulomb parameter e:      {popt[2]:.4f} (Lüscher ref: pi/12 = 0.2618)")
    print("-" * 75)
    print("  GEVP Stability Spectrum:")
    print(f"    Primary Hybrid   [0,4,8] (t0=1->t=2): m*a = {m_hyb_12:.4f} +/- {m_hyb_err:.4f} | cond={cond_h:.1f} ({ng_h12}/3 modes)")
    print(f"    Primary Hybrid   [0,4,8] (t0=2->t=3): m*a = {m_hyb_23:.4f} (noise/plateau audit)")
    print(f"    Pure Smearing    [0,3,6] (t0=1->t=2): m*a = {m_sm_12:.4f} | cond={cond_sm:.1f} ({ng_sm}/3 modes)")
    print(f"    Pure Geometry    [6,7,8] (t0=1->t=2): m*a = {m_gm_12:.4f} | cond={cond_gm:.1f} ({ng_gm}/3 modes)")
    print("-" * 75)
    print(f"  MASS RATIO R0 = m / √σ:   {R_0:.3f} +/- {R_0_err:.3f}")
    print("  Expected Physical Range:  3.40 - 3.75")
    print("="*75)
    print(f"  DATA SHA-256:     {data_hash}")
    print(f"  MANIFEST SHA-256: {manifest_hash}")
    print("="*75)

if __name__ == "__main__":
    run_point_beta25_L16()

In [ ]:
import numpy as np

# 1. Загрузка данных
fn = 'su2_production_beta25_L16.npz'
d = np.load(fn)
O_data = d['O_data']
sigma_a2 = 0.04545
sigma_err = 0.00078

L = 16
n_meas, n_ops, _ = O_data.shape
n_bins = 40
bin_size = n_meas // n_bins
n_use = n_bins * bin_size

# 2. Корреляторы и биннинг
G = np.zeros((n_meas, n_ops, n_ops, L), dtype=np.float64)
for dt in range(L):
  G[:, :, :, dt] = (
      np.einsum('mit,mjt->mij', O_data, np.roll(O_data, -dt, axis=-1)) / L
  )
Om = O_data.mean(axis=-1)

Gb = G[:n_use].reshape(n_bins, bin_size, n_ops, n_ops, L).mean(axis=1)
Ob = Om[:n_use].reshape(n_bins, bin_size, n_ops).mean(axis=1)


def build_C(G_in, Om_in):
  C = G_in - np.outer(Om_in, Om_in)[:, :, None]
  C = 0.5 * (C + C.transpose(1, 0, 2))
  Cs = np.zeros_like(C)
  Cs[:, :, 0] = C[:, :, 0]
  for dt in range(1, L):
    Cs[:, :, dt] = 0.5 * (C[:, :, dt] + C[:, :, L - dt])
  return Cs


C_full = build_C(Gb.mean(0), Ob.mean(0))


def gevp(C_mat, t0, t_tgt, rcond=1e-3):
  C0 = 0.5 * (C_mat[:, :, t0] + C_mat[:, :, t0].T)
  w0, v0 = np.linalg.eigh(C0)
  max_w = w0.max()
  if max_w <= 0:
    return np.nan
  good = w0 > (rcond * max_w)
  if good.sum() < 2:
    return np.nan
  inv = v0[:, good] @ np.diag(1.0 / np.sqrt(w0[good]))
  Ct = 0.5 * (C_mat[:, :, t_tgt] + C_mat[:, :, t_tgt].T)
  w = np.linalg.eigvalsh(inv.T @ Ct @ inv)
  w = w[np.isfinite(w) & (w > 0)]
  return -np.log(w.max()) / (t_tgt - t0) if len(w) else np.nan


# 3. Базис [0, 4, 8] = {1x1_A2, 1x2_A6, 1x3_A12}
basis = [0, 4, 8]
Cb = C_full[np.ix_(basis, basis)]

m_12 = gevp(Cb, 1, 2)
m_23 = gevp(Cb, 2, 3)
m_34 = gevp(Cb, 3, 4)

# 4. Jackknife по бинам
m_12_jk = np.full(n_bins, np.nan)
m_23_jk = np.full(n_bins, np.nan)
m_34_jk = np.full(n_bins, np.nan)

for b in range(n_bins):
  Gsub = np.delete(Gb, b, axis=0).mean(0)
  Osub = np.delete(Ob, b, axis=0).mean(0)
  Csub = build_C(Gsub, Osub)[np.ix_(basis, basis)]
  m_12_jk[b] = gevp(Csub, 1, 2)
  m_23_jk[b] = gevp(Csub, 2, 3)
  m_34_jk[b] = gevp(Csub, 3, 4)


def jk_err(a):
  v = a[np.isfinite(a)]
  return (
      np.sqrt((len(v) - 1) / len(v) * np.sum((v - v.mean()) ** 2))
      if len(v) > 1
      else np.nan
  )


err_12 = jk_err(m_12_jk)
err_23 = jk_err(m_23_jk)
err_34 = jk_err(m_34_jk)


def compute_R0(m, me, s, se):
  if not (np.isfinite(m) and m > 0):
    return np.nan, np.nan
  r = m / np.sqrt(s)
  re = r * np.sqrt((me / m) ** 2 + 0.25 * (se / s) ** 2)
  return r, re


R_12, R_12_err = compute_R0(m_12, err_12, sigma_a2, sigma_err)
R_23, R_23_err = compute_R0(m_23, err_23, sigma_a2, sigma_err)
R_34, R_34_err = compute_R0(m_34, err_34, sigma_a2, sigma_err)

print('=' * 72)
print('РЕАНАЛИЗ СПЕКТРА БЕТА = 2.5 (L = 16^4, N = 2000)')
print('=' * 72)
print(
    f'  m(t=1->2)*a = {m_12:.4f} +/- {err_12:.4f}  |  R_0(1->2) ='
    f' {R_12:.3f} +/- {R_12_err:.3f}'
)
print(
    f'  m(t=2->3)*a = {m_23:.4f} +/- {err_23:.4f}  |  R_0(2->3) ='
    f' {R_23:.3f} +/- {R_23_err:.3f}'
)
print(
    f'  m(t=3->4)*a = {m_34:.4f} +/- {err_34:.4f}  |  R_0(3->4) ='
    f' {R_34:.3f} +/- {R_34_err:.3f}'
)
print('=' * 72)

In [ ]:
import numpy as np

# 1. Загрузка данных
fn = 'su2_production_beta25_L16.npz'
d = np.load(fn)
O_data = d['O_data']
sigma_a2 = 0.04545
sigma_err = 0.00078

L = 16
n_meas, n_ops, _ = O_data.shape
n_bins = 40
bin_size = n_meas // n_bins
n_use = n_bins * bin_size

# 2. Корреляторы и биннинг
G = np.zeros((n_meas, n_ops, n_ops, L), dtype=np.float64)
for dt in range(L):
  G[:, :, :, dt] = (
      np.einsum('mit,mjt->mij', O_data, np.roll(O_data, -dt, axis=-1)) / L
  )
Om = O_data.mean(axis=-1)

Gb = G[:n_use].reshape(n_bins, bin_size, n_ops, n_ops, L).mean(axis=1)
Ob = Om[:n_use].reshape(n_bins, bin_size, n_ops).mean(axis=1)


def build_C(G_in, Om_in):
  C = G_in - np.outer(Om_in, Om_in)[:, :, None]
  C = 0.5 * (C + C.transpose(1, 0, 2))
  Cs = np.zeros_like(C)
  Cs[:, :, 0] = C[:, :, 0]
  for dt in range(1, L):
    Cs[:, :, dt] = 0.5 * (C[:, :, dt] + C[:, :, L - dt])
  return Cs


C_full = build_C(Gb.mean(0), Ob.mean(0))


def gevp(C_mat, t0, t_tgt, rcond=1e-3):
  C0 = 0.5 * (C_mat[:, :, t0] + C_mat[:, :, t0].T)
  w0, v0 = np.linalg.eigh(C0)
  max_w = w0.max()
  if max_w <= 0:
    return np.nan
  good = w0 > (rcond * max_w)
  if good.sum() < 2:
    return np.nan
  inv = v0[:, good] @ np.diag(1.0 / np.sqrt(w0[good]))
  Ct = 0.5 * (C_mat[:, :, t_tgt] + C_mat[:, :, t_tgt].T)
  w = np.linalg.eigvalsh(inv.T @ Ct @ inv)
  w = w[np.isfinite(w) & (w > 0)]
  return -np.log(w.max()) / (t_tgt - t0) if len(w) else np.nan


# 3. Базис [0, 4, 8] = {1x1_A2, 1x2_A6, 1x3_A12}
basis = [0, 4, 8]
Cb = C_full[np.ix_(basis, basis)]

m_12 = gevp(Cb, 1, 2)
m_23 = gevp(Cb, 2, 3)
m_34 = gevp(Cb, 3, 4)

# 4. Jackknife по бинам
m_12_jk = np.full(n_bins, np.nan)
m_23_jk = np.full(n_bins, np.nan)
m_34_jk = np.full(n_bins, np.nan)

for b in range(n_bins):
  Gsub = np.delete(Gb, b, axis=0).mean(0)
  Osub = np.delete(Ob, b, axis=0).mean(0)
  Csub = build_C(Gsub, Osub)[np.ix_(basis, basis)]
  m_12_jk[b] = gevp(Csub, 1, 2)
  m_23_jk[b] = gevp(Csub, 2, 3)
  m_34_jk[b] = gevp(Csub, 3, 4)


def jk_err(a):
  v = a[np.isfinite(a)]
  return (
      np.sqrt((len(v) - 1) / len(v) * np.sum((v - v.mean()) ** 2))
      if len(v) > 1
      else np.nan
  )


err_12 = jk_err(m_12_jk)
err_23 = jk_err(m_23_jk)
err_34 = jk_err(m_34_jk)


def compute_R0(m, me, s, se):
  if not (np.isfinite(m) and m > 0):
    return np.nan, np.nan
  r = m / np.sqrt(s)
  re = r * np.sqrt((me / m) ** 2 + 0.25 * (se / s) ** 2)
  return r, re


R_12, R_12_err = compute_R0(m_12, err_12, sigma_a2, sigma_err)
R_23, R_23_err = compute_R0(m_23, err_23, sigma_a2, sigma_err)
R_34, R_34_err = compute_R0(m_34, err_34, sigma_a2, sigma_err)

print('=' * 72)
print('РЕАНАЛИЗ СПЕКТРА БЕТА = 2.5 (L = 16^4, N = 2000)')
print('=' * 72)
print(
    f'  m(t=1->2)*a = {m_12:.4f} +/- {err_12:.4f}  |  R_0(1->2) ='
    f' {R_12:.3f} +/- {R_12_err:.3f}'
)
print(
    f'  m(t=2->3)*a = {m_23:.4f} +/- {err_23:.4f}  |  R_0(2->3) ='
    f' {R_23:.3f} +/- {R_23_err:.3f}'
)
print(
    f'  m(t=3->4)*a = {m_34:.4f} +/- {err_34:.4f}  |  R_0(3->4) ='
    f' {R_34:.3f} +/- {R_34_err:.3f}'
)
print('=' * 72)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

# База данных ансамблей
data = {
    'b2.3_L12': {
        'beta': 2.3,
        'L': 12,
        'a2s': 0.14214,
        'a2s_err': 0.00839,
        'ma': 1.2285,
        'ma_err': 0.0716,
    },
    'b2.4_L12': {
        'beta': 2.4,
        'L': 12,
        'a2s': 0.08195,
        'a2s_err': 0.00268,
        'ma': 0.9712,
        'ma_err': 0.0691,
    },
    'b2.4_L16': {
        'beta': 2.4,
        'L': 16,
        'a2s': 0.08195,  # подставить sigma_a2 из вывода Скрипта 2
        'a2s_err': 0.00268,  # подставить sigma_err из вывода Скрипта 2
        'ma': None,  # вписать m*a из вывода Скрипта 2 (t=1->2 или t=2->3)
        'ma_err': None,  # вписать m_err из вывода Скрипта 2
    },
    'b2.5_L16': {
        'beta': 2.5,
        'L': 16,
        'a2s': 0.04545,
        'a2s_err': 0.00078,
        'ma': 0.8045,  # значение из Скрипта 1 (срез t=2->3)
        'ma_err': 0.0918,  # ошибка Jackknife из Скрипта 1
    },
}


def compute_R0(m, m_err, s, s_err):
  R = m / np.sqrt(s)
  R_err = R * np.sqrt((m_err / m) ** 2 + 0.25 * (s_err / s) ** 2)
  return R, R_err


points = []
for key, d in data.items():
  if d['ma'] is None or d['a2s'] is None:
    continue
  R, Re = compute_R0(d['ma'], d['ma_err'], d['a2s'], d['a2s_err'])
  points.append((d['a2s'], R, Re, key))

points.sort(key=lambda item: item[0])
x = np.array([p[0] for p in points])
y = np.array([p[1] for p in points])
yerr = np.array([p[2] for p in points])

# Линейный фит
def linear(x, c0, c1):
  return c0 + c1 * x


popt, pcov = curve_fit(linear, x, y, sigma=yerr, absolute_sigma=True)
c0, c1 = popt
dc0, dc1 = np.sqrt(np.diag(pcov))

# Проверка chi2 / dof
resid = y - linear(x, *popt)
chi2 = np.sum((resid / yerr) ** 2)
dof = len(x) - 2
chi2_dof = chi2 / dof if dof > 0 else np.nan

# Квадратичный фит для систематики (если точек >= 4)
if len(x) >= 4:

  def quad(x, c0, c1, c2):
    return c0 + c1 * x + c2 * x**2

  popt_q, _ = curve_fit(
      quad, x, y, sigma=yerr, absolute_sigma=True, maxfev=15000
  )
  c0_q = popt_q[0]
  sys_fit = abs(c0 - c0_q)
else:
  c0_q = np.nan
  sys_fit = np.nan

print('=' * 65)
print('РЕЗУЛЬТАТЫ КОНТИНУАЛЬНОЙ ЭКСТРАПОЛЯЦИИ')
print('=' * 65)
print(f'Использовано ансамблей: {len(x)}')
print(f'Линейный фит:    R_0(cont) = {c0:.3f} +/- {dc0:.3f}')
print(f'Наклон Симанзика: c_1       = {c1:.3f} +/- {dc1:.3f}')
print(f'Качество фита:   chi2/dof  = {chi2_dof:.2f} (chi2={chi2:.2f}, dof={dof})')
if np.isfinite(sys_fit):
  print(f'Квадратичный:    R_0(cont) = {c0_q:.3f}')
  print(f'Систематика фита: delta_sys = {sys_fit:.3f}')
print('=' * 65)

# Построение графика
plt.figure(figsize=(8, 5.5))
xg = np.linspace(0, x.max() * 1.15, 200)

plt.errorbar(
    x,
    y,
    yerr=yerr,
    fmt='o',
    color='black',
    ecolor='black',
    elinewidth=1.5,
    capsize=5,
    label='Lattice Data',
    zorder=5,
)
plt.plot(
    xg,
    linear(xg, *popt),
    'b-',
    lw=2,
    label=rf'Linear Symanzik: $R_0(0) = {c0:.3f} \pm {dc0:.3f}$',
)
plt.fill_between(
    xg,
    linear(xg, *popt) - dc0,
    linear(xg, *popt) + dc0,
    color='blue',
    alpha=0.15,
    label=r'$1\sigma$ Confidence Interval',
)

# Бенчмарк Майкла Тепера
plt.axhline(
    3.78,
    color='red',
    linestyle='--',
    alpha=0.7,
    label='Teper Benchmark SU(2): 3.78 +/- 0.16',
)
plt.fill_between(
    xg, 3.78 - 0.16, 3.78 + 0.16, color='red', alpha=0.08
)

plt.xlabel(r'$a^2\sigma$', fontsize=13)
plt.ylabel(r'$m_{0^{++}} / \sqrt{\sigma}$', fontsize=13)
plt.xlim(0, xg.max())
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(frameon=True, fontsize=10)
plt.tight_layout()

plt.savefig('continuum_extrapolation.pdf')
plt.savefig('continuum_extrapolation.png', dpi=300)
print(
    'Графики сохранены: continuum_extrapolation.pdf,'
    ' continuum_extrapolation.png'
)

In [ ]:
import os
import glob
import numpy as np

# 1. Спецификация ансамблей
ensembles = [
    {
        'pattern': '*beta23_L12.npz',
        'beta': 2.3,
        'L': 12,
        'sigma': 0.14214,
        'sigma_err': 0.00839,
        'n_bins': 40,
        'label': 'beta=2.3 (L=12^4)'
    },
    {
        'pattern': '*beta24*ape2_6_12.npz',
        'beta': 2.4,
        'L': 12,
        'sigma': 0.08195,
        'sigma_err': 0.00268,
        'n_bins': 50,
        'label': 'beta=2.4 (L=12^4)'
    },
    {
        'pattern': '*beta25_L16.npz',
        'beta': 2.5,
        'L': 16,
        'sigma': 0.04545,
        'sigma_err': 0.00078,
        'n_bins': 40,
        'label': 'beta=2.5 (L=16^4)'
    }
]

def find_file(pattern):
    candidates = glob.glob(f"/kaggle/working/**/{pattern}", recursive=True) + \
                 glob.glob(f"/kaggle/input/**/{pattern}", recursive=True) + \
                 glob.glob(pattern)
    return candidates[0] if candidates else None

def build_C(G, Om, L):
    C = G - np.outer(Om, Om)[:, :, None]
    C = 0.5 * (C + C.transpose(1, 0, 2))
    Cs = np.zeros_like(C)
    Cs[:, :, 0] = C[:, :, 0]
    for dt in range(1, L):
        Cs[:, :, dt] = 0.5 * (C[:, :, dt] + C[:, :, L - dt])
    return Cs

def gevp_solve(C_mat, t0, t_tgt, rcond=1e-3):
    C0 = 0.5 * (C_mat[:, :, t0] + C_mat[:, :, t0].T)
    w0, v0 = np.linalg.eigh(C0)
    max_w = w0.max()
    if max_w <= 0: return np.nan
    good = w0 > (rcond * max_w)
    if good.sum() < 2: return np.nan
    inv = v0[:, good] @ np.diag(1.0 / np.sqrt(w0[good]))
    Ct = 0.5 * (C_mat[:, :, t_tgt] + C_mat[:, :, t_tgt].T)
    w = np.linalg.eigvalsh(inv.T @ Ct @ inv)
    w = w[np.isfinite(w) & (w > 0)]
    return -np.log(w.max()) / (t_tgt - t0) if len(w) > 0 else np.nan

def jk_error(arr):
    v = arr[np.isfinite(arr)]
    if len(v) < 2: return np.nan
    return np.sqrt((len(v) - 1) / len(v) * np.sum((v - v.mean())**2))

def compute_R0_full(m, me, s, se):
    if not (np.isfinite(m) and m > 0): return np.nan, np.nan
    r = m / np.sqrt(s)
    re = r * np.sqrt((me / m)**2 + 0.25 * (se / s)**2)
    return r, re

# ==============================================================================
# 2. Пакетный расчёт
# ==============================================================================
basis = [0, 4, 8]  # Hybrid basis: {1x1_A2, 1x2_A6, 1x3_A12}
results_table = []

print("=" * 82)
print("ПАКЕТНЫЙ РЕАНАЛИЗ БАЗИСА GEVP: ОКНА t=1->2 vs t=2->3")
print("=" * 82)

for ens in ensembles:
    fp = find_file(ens['pattern'])
    if not fp:
        print(f"[!] Файл для {ens['label']} не найден (проверьте наличие на диске).")
        continue

    d = np.load(fp)
    O_data = d['O_data']
    L = ens['L']
    n_meas = O_data.shape[0]
    n_ops = O_data.shape[1]
    n_bins = ens['n_bins']
    bin_size = n_meas // n_bins
    n_use = n_bins * bin_size

    # Свёртка корреляторов
    G = np.zeros((n_meas, n_ops, n_ops, L), dtype=np.float64)
    for dt in range(L):
        G[:, :, :, dt] = np.einsum('mit,mjt->mij', O_data, np.roll(O_data, -dt, axis=-1)) / L
    Om = O_data.mean(axis=-1)

    Gb = G[:n_use].reshape(n_bins, bin_size, n_ops, n_ops, L).mean(axis=1)
    Ob = Om[:n_use].reshape(n_bins, bin_size, n_ops).mean(axis=1)

    # Полная оценка
    C_full = build_C(Gb.mean(0), Ob.mean(0), L)
    Cb = C_full[np.ix_(basis, basis)]
    m12 = gevp_solve(Cb, 1, 2)
    m23 = gevp_solve(Cb, 2, 3)

    # Jackknife ресемплинг
    m12_jk = np.full(n_bins, np.nan)
    m23_jk = np.full(n_bins, np.nan)

    for b in range(n_bins):
        Gsub = np.delete(Gb, b, axis=0).mean(0)
        Osub = np.delete(Ob, b, axis=0).mean(0)
        Csub = build_C(Gsub, Osub, L)[np.ix_(basis, basis)]
        m12_jk[b] = gevp_solve(Csub, 1, 2)
        m23_jk[b] = gevp_solve(Csub, 2, 3)

    m12_err = jk_error(m12_jk)
    m23_err = jk_error(m23_jk)

    r12, r12_e = compute_R0_full(m12, m12_err, ens['sigma'], ens['sigma_err'])
    r23, r23_e = compute_R0_full(m23, m23_err, ens['sigma'], ens['sigma_err'])

    results_table.append({
        'label': ens['label'],
        'beta': ens['beta'],
        'm12': m12, 'm12_err': m12_err, 'r12': r12, 'r12_e': r12_e,
        'm23': m23, 'm23_err': m23_err, 'r23': r23, 'r23_e': r23_e
    })

# Вывод сводной таблицы 3x2
print(f"{'Ансамбль':<18} | {'Окно t=1->2: m*a':<18} | {'R_0 (t=1->2)':<16} | {'Окно t=2->3: m*a':<18} | {'R_0 (t=2->3)':<16}")
print("-" * 96)
for r in results_table:
    s_m12 = f"{r['m12']:.4f} +/- {r['m12_err']:.4f}" if np.isfinite(r['m12']) else "NaN"
    s_r12 = f"{r['r12']:.3f} +/- {r['r12_e']:.3f}" if np.isfinite(r['r12']) else "NaN"
    s_m23 = f"{r['m23']:.4f} +/- {r['m23_err']:.4f}" if np.isfinite(r['m23']) else "NaN"
    s_r23 = f"{r['r23']:.3f} +/- {r['r23_e']:.3f}" if np.isfinite(r['r23']) else "NaN"
    print(f"{r['label']:<18} | {s_m12:<18} | {s_r12:<16} | {s_m23:<18} | {s_r23:<16}")
print("=" * 96)

In [ ]:
# ==============================================================================
# SU(2) Glueball GEVP + Cornell Potential: beta=2.4, L=16^4 (Isovolumetric Run)
# Basis: Hybrid [0,4,8], Pure-Smear [0,3,6], Pure-Geometry [6,7,8]
# Dual-Plateau Jackknife (t=1->2 & t=2->3), Safe Cornell, SHA-256 Manifest
# ==============================================================================
import torch
import numpy as np
from scipy.optimize import curve_fit
import hashlib, json, time, os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active Compute Device: {device}")

# 1. Алгебра кватернионов SU(2)
def qmul(a, b):
    a0, a1, a2, a3 = a.unbind(-1)
    b0, b1, b2, b3 = b.unbind(-1)
    return torch.stack([
        a0*b0 - a1*b1 - a2*b2 - a3*b3,
        a0*b1 + a1*b0 + a2*b3 - a3*b2,
        a0*b2 - a1*b3 + a2*b0 + a3*b1,
        a0*b3 + a1*b2 - a2*b1 + a3*b0
    ], dim=-1)

def qdag(q):
    return q * q.new_tensor([1., -1., -1., -1.])

def qnorm(q):
    return q / (torch.norm(q, dim=-1, keepdim=True).clamp(min=1e-14))

# 2. Стапели с корректной геометрией контура
def get_staples(U, mu):
    S = torch.zeros_like(U[mu])
    for nu in range(4):
        if nu == mu: continue
        s1 = qmul(
            torch.roll(U[nu], shifts=-1, dims=mu),
            qmul(qdag(torch.roll(U[mu], shifts=-1, dims=nu)), qdag(U[nu]))
        )
        u_nu_bwd = torch.roll(U[nu], shifts=+1, dims=nu)
        s2 = qmul(
            qdag(torch.roll(u_nu_bwd, shifts=-1, dims=mu)),
            qmul(qdag(torch.roll(U[mu], shifts=+1, dims=nu)), u_nu_bwd)
        )
        S = S + s1 + s2
    return S

# 3. Kennedy-Pendleton Heatbath & Overrelaxation
def heatbath_sweep(U, beta, parity_mask):
    for mu in range(4):
        for p in (0, 1):
            mask = (parity_mask == p)
            S = get_staples(U, mu)
            S_m = S[mask]
            s_norm = torch.norm(S_m, dim=-1, keepdim=True).clamp(min=1e-12)
            V = S_m / s_norm
            alpha = (beta * s_norm).squeeze(-1)

            x = torch.zeros_like(alpha)
            accepted = torch.zeros(alpha.shape[0], dtype=torch.bool, device=device)
            for _ in range(12):
                r1 = torch.rand_like(alpha).clamp(min=1e-12)
                r2 = torch.rand_like(alpha).clamp(min=1e-12)
                r3 = torch.rand_like(alpha)
                r4 = torch.rand_like(alpha)
                cand_x = (-torch.log(r1) - torch.log(r2) * (torch.cos(2.0 * np.pi * r3)**2)) / alpha
                ok = (cand_x <= 2.0) & (r4**2 <= (1.0 - 0.5 * cand_x))
                new_acc = (~accepted) & ok
                x = torch.where(new_acc, cand_x, x)
                accepted = accepted | ok

            x = torch.where(accepted, x, (0.5 / alpha).clamp(max=2.0))
            q0 = (1.0 - x).unsqueeze(-1)
            phi = 2.0 * np.pi * torch.rand_like(q0)
            ct = 2.0 * torch.rand_like(q0) - 1.0
            st = (1.0 - ct**2).clamp(min=0.0).sqrt()
            r = (1.0 - q0**2).clamp(min=0.0).sqrt()
            qv = torch.cat([q0, r*st*torch.cos(phi), r*st*torch.sin(phi), r*ct], dim=-1)
            u_new = qmul(qv, qdag(V))
            U[mu, mask] = qnorm(u_new)
    return U

def overrelax_sweep(U, parity_mask):
    for mu in range(4):
        for p in (0, 1):
            mask = (parity_mask == p)
            S = get_staples(U, mu)
            S_m = S[mask]
            s_norm = torch.norm(S_m, dim=-1, keepdim=True).clamp(min=1e-12)
            V = S_m / s_norm
            V_dag = qdag(V)
            U_old = U[mu, mask]
            u_new = qmul(V_dag, qmul(qdag(U_old), V_dag))
            U[mu, mask] = qnorm(u_new)
    return U

# 4. Пространственное APE-сглаживание
def ape_step_spatial(U, alpha=0.20):
    U_new = U.clone()
    for mu in (1, 2, 3):
        C = torch.zeros_like(U[mu])
        for nu in (1, 2, 3):
            if nu == mu: continue
            u1 = U[nu]
            u2 = torch.roll(U[mu], shifts=-1, dims=nu)
            u3 = qdag(torch.roll(U[nu], shifts=-1, dims=mu))
            c_pos = qmul(u1, qmul(u2, u3))

            u_nu_bwd = torch.roll(U[nu], shifts=+1, dims=nu)
            u1_neg = qdag(u_nu_bwd)
            u2_neg = torch.roll(U[mu], shifts=+1, dims=nu)
            u3_neg = torch.roll(u_nu_bwd, shifts=-1, dims=mu)
            c_neg = qmul(u1_neg, qmul(u2_neg, u3_neg))

            C = C + c_pos + c_neg
        W = (1.0 - alpha) * U[mu] + (alpha / 4.0) * C
        U_new[mu] = qnorm(W)
    return U_new

# 5. Операторы и петли Вильсона
def build_rect_loop(U_field, mu, nu, R=1, T=1):
    acc_mu = U_field[mu]
    for i in range(1, R):
        acc_mu = qmul(acc_mu, torch.roll(U_field[mu], shifts=-i, dims=mu))
    acc_nu = U_field[nu]
    for j in range(1, T):
        acc_nu = qmul(acc_nu, torch.roll(U_field[nu], shifts=-j, dims=nu))

    nu_shift_R = torch.roll(acc_nu, shifts=-R, dims=mu)
    mu_shift_T = torch.roll(acc_mu, shifts=-T, dims=nu)
    c12 = qmul(acc_mu, nu_shift_R)
    c34 = qmul(qdag(mu_shift_T), qdag(acc_nu))
    return qmul(c12, c34)

def measure_operators(U_sm):
    L = U_sm.shape[1]
    O_1x1 = torch.zeros((L, L, L, L), device=device)
    O_1x2 = torch.zeros((L, L, L, L), device=device)
    O_1x3 = torch.zeros((L, L, L, L), device=device)

    planes = [(1, 2), (1, 3), (2, 3)]
    for mu, nu in planes:
        P = build_rect_loop(U_sm, mu, nu, R=1, T=1)
        O_1x1 = O_1x1 + P[..., 0]

        W12 = build_rect_loop(U_sm, mu, nu, R=1, T=2)
        W21 = build_rect_loop(U_sm, nu, mu, R=1, T=2)
        O_1x2 = O_1x2 + W12[..., 0] + W21[..., 0]

        W13 = build_rect_loop(U_sm, mu, nu, R=1, T=3)
        W31 = build_rect_loop(U_sm, nu, mu, R=1, T=3)
        O_1x3 = O_1x3 + W13[..., 0] + W31[..., 0]

    return (O_1x1 / 3.0).mean(dim=(1, 2, 3)), (O_1x2 / 6.0).mean(dim=(1, 2, 3)), (O_1x3 / 6.0).mean(dim=(1, 2, 3))

def measure_wilson_loops(U_hyb, R_values=(1, 2, 3, 4, 5), T_values=(1, 2, 3, 4, 5)):
    def path_spatial(k, R):
        acc = U_hyb[k]
        for i in range(1, R):
            acc = qmul(acc, torch.roll(U_hyb[k], shifts=-i, dims=k))
        return acc

    def path_temporal(T):
        acc = U_hyb[0]
        for j in range(1, T):
            acc = qmul(acc, torch.roll(U_hyb[0], shifts=-j, dims=0))
        return acc

    results = {}
    for R in R_values:
        for T in T_values:
            val = 0.0
            for k in (1, 2, 3):
                Lk = path_spatial(k, R)
                T0 = path_temporal(T)
                T0_shift_R = torch.roll(T0, shifts=-R, dims=k)
                Lk_shift_T = torch.roll(Lk, shifts=-T, dims=0)
                c12 = qmul(Lk, T0_shift_R)
                c34 = qmul(qdag(Lk_shift_T), qdag(T0))
                loop = qmul(c12, c34)
                val += loop[..., 0].mean().item()
            results[(R, T)] = val / 3.0
    return results

# 6. GEVP и численный анализ
def gevp_solve(C_mat, t0=1, t_target=2, rcond=1e-3):
    C0 = 0.5 * (C_mat[:, :, t0] + C_mat[:, :, t0].T)
    w0, v0 = np.linalg.eigh(C0)
    max_w = w0.max()
    if max_w <= 0: return np.nan, 0, np.inf
    good = w0 > (rcond * max_w)
    pos_w = w0[w0 > 0]
    cond = max_w / max(pos_w.min(), 1e-16) if len(pos_w) > 0 else np.inf
    if np.sum(good) < 2: return np.nan, int(np.sum(good)), cond

    inv_sqrt = v0[:, good] @ np.diag(1.0 / np.sqrt(w0[good]))
    Ct = 0.5 * (C_mat[:, :, t_target] + C_mat[:, :, t_target].T)
    C_tilde = inv_sqrt.T @ Ct @ inv_sqrt
    w_t = np.linalg.eigvalsh(C_tilde)
    w_pos = w_t[np.isfinite(w_t) & (w_t > 0)]
    if len(w_pos) == 0: return np.nan, int(np.sum(good)), cond
    lam = np.max(w_pos)
    return -np.log(lam) / (t_target - t0), int(np.sum(good)), cond

def build_C(G, O_mean):
    C = G - np.outer(O_mean, O_mean)[:, :, None]
    C = 0.5 * (C + C.transpose(1, 0, 2))
    L = C.shape[-1]
    Cs = np.zeros_like(C)
    Cs[:, :, 0] = C[:, :, 0]
    for dt in range(1, L):
        Cs[:, :, dt] = 0.5 * (C[:, :, dt] + C[:, :, L - dt])
    return Cs

def cornell(R, V0, sigma, e):
    return V0 + sigma * R - e / R

def safe_fit_cornell(V_arr, R_arr):
    mask = (R_arr >= 2) & np.isfinite(V_arr)
    if np.sum(mask) < 3: return np.array([np.nan, np.nan, np.nan])
    try:
        popt, _ = curve_fit(
            cornell, R_arr[mask], V_arr[mask],
            p0=[0.53, 0.082, 0.26],
            bounds=([-2.0, 0.03, 0.0], [3.0, 0.25, 1.0]),
            maxfev=15000
        )
        return popt
    except Exception:
        return np.array([np.nan, np.nan, np.nan])

def calc_jk_err(arr):
    v = arr[np.isfinite(arr)]
    if len(v) < 2: return np.nan
    return np.sqrt((len(v) - 1) / len(v) * np.sum((v - v.mean())**2))

def safe_log_ratio(a, b):
    if np.isfinite(a) and np.isfinite(b) and a > 0 and b > 0:
        return np.log(a / b)
    return np.nan

def get_sha256(path):
    sha = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(8192 * 1024):
            sha.update(chunk)
    return sha.hexdigest()

# 7. Конвейер симуляции
def run_point_beta24_L16(L=16, beta=2.4, n_therm=100, n_meas=2000,
                         n_overrelax=4, n_bins=40,
                         ape_levels=(2, 6, 12), ape_alpha=0.20, loop_smear=6,
                         R_values=(1, 2, 3, 4, 5), T_values=(1, 2, 3, 4, 5)):

    print("="*75)
    print(f"SU(2) GLUEBALL SPECTROSCOPY: beta={beta}, L={L}^4 (ISOVOLUMETRIC CHECK)")
    print(f"Statistics: {n_meas} configs, {n_bins} bins ({n_meas//n_bins} cfg/bin), OR/HB: {n_overrelax}")
    print(f"APE Scheme: {ape_levels}, alpha={ape_alpha} | loop_smear={loop_smear}")
    print("="*75)

    U = torch.zeros((4, L, L, L, L, 4), device=device)
    U[..., 0] = 1.0
    coords = torch.stack(torch.meshgrid([torch.arange(L, device=device)]*4, indexing='ij'))
    parity = (coords.sum(0)) % 2

    def mean_plaquette(Ux):
        tot, cnt = 0.0, 0
        for mu in range(4):
            for nu in range(mu+1, 4):
                u_nu_fwd = torch.roll(Ux[nu], shifts=-1, dims=mu)
                u_mu_fwd = torch.roll(Ux[mu], shifts=-1, dims=nu)
                P = qmul(Ux[mu], qmul(u_nu_fwd, qmul(qdag(u_mu_fwd), qdag(Ux[nu]))))
                tot += P[..., 0].mean().item()
                cnt += 1
        return tot / cnt

    print(f"[Sanity 1] Cold Start Plaquette: {mean_plaquette(U):.6f}")

    print("\nStage 1: Thermalization...")
    t_start = time.time()
    for step in range(n_therm):
        U = heatbath_sweep(U, beta, parity)
        for _ in range(n_overrelax):
            U = overrelax_sweep(U, parity)
        if (step + 1) % 25 == 0:
            print(f"  Therm {step+1:3d}/{n_therm} | <P> = {mean_plaquette(U):.6f} | {time.time()-t_start:.1f}s")

    n_lev = len(ape_levels)
    n_ops = 3 * n_lev
    O_data = np.zeros((n_meas, n_ops, L), dtype=np.float64)
    plaq_arr = np.zeros(n_meas, dtype=np.float64)
    loop_keys = [(R, T) for R in R_values for T in T_values]
    W_data = np.zeros((n_meas, len(loop_keys)), dtype=np.float64)

    print(f"\nStage 2: Production ({n_meas} trajectories)...")
    t_start = time.time()
    for m in range(n_meas):
        U = heatbath_sweep(U, beta, parity)
        for _ in range(n_overrelax):
            U = overrelax_sweep(U, parity)
        plaq_arr[m] = mean_plaquette(U)

        U_sm = U.clone()
        cur = 0
        for li, target in enumerate(ape_levels):
            while cur < target:
                U_sm = ape_step_spatial(U_sm, ape_alpha)
                cur += 1
            o1, o2, o3 = measure_operators(U_sm)
            O_data[m, 3*li + 0, :] = o1.cpu().numpy()
            O_data[m, 3*li + 1, :] = o2.cpu().numpy()
            O_data[m, 3*li + 2, :] = o3.cpu().numpy()

            if cur == loop_smear:
                U_hyb = U_sm.clone()
                U_hyb[0] = U[0]
                w_dict = measure_wilson_loops(U_hyb, R_values, T_values)
                for ki, k in enumerate(loop_keys):
                    W_data[m, ki] = w_dict[k]

        if (m + 1) % 100 == 0:
            print(f"  Config {m+1:4d}/{n_meas} | Running <P> = {plaq_arr[:m+1].mean():.6f} | {time.time()-t_start:.1f}s")

    fn_data = 'su2_production_beta24_L16.npz'
    np.savez(fn_data, O_data=O_data, W_data=W_data, plaq_arr=plaq_arr)
    print(f"  [Auto-Save] Конфигурации сохранены в '{fn_data}'")

    bin_size = n_meas // n_bins
    n_use = n_bins * bin_size

    G_cfg = np.zeros((n_meas, n_ops, n_ops, L), dtype=np.float64)
    for dt in range(L):
        G_cfg[:, :, :, dt] = np.einsum('mit,mjt->mij', O_data, np.roll(O_data, -dt, axis=-1)) / L
    O_mean_cfg = O_data.mean(axis=-1)

    G_binned = G_cfg[:n_use].reshape(n_bins, bin_size, n_ops, n_ops, L).mean(axis=1)
    O_binned = O_mean_cfg[:n_use].reshape(n_bins, bin_size, n_ops).mean(axis=1)
    W_binned = W_data[:n_use].reshape(n_bins, bin_size, len(loop_keys)).mean(axis=1)

    C_full = build_C(G_binned.mean(0), O_binned.mean(0))
    C_jk_all = np.zeros((n_bins, n_ops, n_ops, L), dtype=np.float64)
    for b in range(n_bins):
        Gb = np.delete(G_binned, b, axis=0).mean(axis=0)
        Ob = np.delete(O_binned, b, axis=0).mean(axis=0)
        C_jk_all[b] = build_C(Gb, Ob)

    # Cornell fit
    W_mean_full = {k: W_binned.mean(0)[ki] for ki, k in enumerate(loop_keys)}
    R_arr = np.array(R_values)
    V_T3 = np.array([safe_log_ratio(W_mean_full[(R, 3)], W_mean_full[(R, 4)]) for R in R_values])
    popt = safe_fit_cornell(V_T3, R_arr)
    sigma_a2 = popt[1]

    sigma_jk = np.full(n_bins, np.nan)
    for b in range(n_bins):
        W_sub = np.delete(W_binned, b, axis=0).mean(axis=0)
        W_map = {k: W_sub[ki] for ki, k in enumerate(loop_keys)}
        Vb = np.array([safe_log_ratio(W_map[(R, 3)], W_map[(R, 4)]) for R in R_values])
        popt_b = safe_fit_cornell(Vb, R_arr)
        sigma_jk[b] = popt_b[1]
    sigma_err = calc_jk_err(sigma_jk)

    # GEVP t=1->2 и t=2->3
    b_hybrid = [0, 4, 8]
    m_hyb_12, ng_12, cond_12 = gevp_solve(C_full[np.ix_(b_hybrid, b_hybrid)], t0=1, t_target=2)
    m_hyb_23, ng_23, cond_23 = gevp_solve(C_full[np.ix_(b_hybrid, b_hybrid)], t0=2, t_target=3)

    m12_jk = np.full(n_bins, np.nan)
    m23_jk = np.full(n_bins, np.nan)
    for b in range(n_bins):
        m12_jk[b], _, _ = gevp_solve(C_jk_all[b][np.ix_(b_hybrid, b_hybrid)], t0=1, t_target=2)
        m23_jk[b], _, _ = gevp_solve(C_jk_all[b][np.ix_(b_hybrid, b_hybrid)], t0=2, t_target=3)

    m12_err = calc_jk_err(m12_jk)
    m23_err = calc_jk_err(m23_jk)

    # Контрольные базисы
    m_sm_12, _, cond_sm = gevp_solve(C_full[np.ix_([0, 3, 6], [0, 3, 6])], t0=1, t_target=2)
    m_gm_12, _, cond_gm = gevp_solve(C_full[np.ix_([6, 7, 8], [6, 7, 8])], t0=1, t_target=2)

    # Отношения масс с Jackknife-ошибками
    R12_jk = np.array([
        m12_jk[b] / np.sqrt(sigma_jk[b])
        for b in range(n_bins)
        if np.isfinite(m12_jk[b]) and np.isfinite(sigma_jk[b]) and m12_jk[b] > 0 and sigma_jk[b] > 0
    ])
    R_0_12 = m_hyb_12 / np.sqrt(sigma_a2) if (np.isfinite(m_hyb_12) and np.isfinite(sigma_a2) and m_hyb_12 > 0 and sigma_a2 > 0) else np.nan
    R_0_12_err = calc_jk_err(R12_jk)

    R23_jk = np.array([
        m23_jk[b] / np.sqrt(sigma_jk[b])
        for b in range(n_bins)
        if np.isfinite(m23_jk[b]) and np.isfinite(sigma_jk[b]) and m23_jk[b] > 0 and sigma_jk[b] > 0
    ])
    R_0_23 = m_hyb_23 / np.sqrt(sigma_a2) if (np.isfinite(m_hyb_23) and np.isfinite(sigma_a2) and m_hyb_23 > 0 and sigma_a2 > 0) else np.nan
    R_0_23_err = calc_jk_err(R23_jk)

    data_hash = get_sha256(fn_data)
    size_mb = os.path.getsize(fn_data) / (1024 * 1024)
    fn_manifest = 'manifest_beta24_L16.json'

    manifest = {
        "experiment": "SU(2) Glueball & String Tension Isovolumetric Point (L=16)",
        "timestamp_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
        "lattice": {"group": "SU(2)", "beta": float(beta), "geometry": f"{L}^4", "action": "Wilson"},
        "monte_carlo": {"n_therm": int(n_therm), "n_meas": int(n_meas), "n_overrelax": int(n_overrelax), "n_bins": int(n_bins)},
        "gevp_diagnostics": {
            "hybrid_t1_t2": {"cond": round(float(cond_12), 1), "modes": f"{ng_12}/3", "m_a": round(float(m_hyb_12), 4), "m_a_err": round(float(m12_err), 4), "R_0": round(float(R_0_12), 3), "R_0_err": round(float(R_0_12_err), 3)},
            "hybrid_t2_t3": {"cond": round(float(cond_23), 1), "modes": f"{ng_23}/3", "m_a": round(float(m_hyb_23), 4) if np.isfinite(m_hyb_23) else None, "m_a_err": round(float(m23_err), 4) if np.isfinite(m23_err) else None, "R_0": round(float(R_0_23), 3) if np.isfinite(R_0_23) else None, "R_0_err": round(float(R_0_23_err), 3) if np.isfinite(R_0_23_err) else None},
            "control_smearing_m_a": round(float(m_sm_12), 4) if np.isfinite(m_sm_12) else None,
            "control_geometry_m_a": round(float(m_gm_12), 4) if np.isfinite(m_gm_12) else None
        },
        "physical_results": {
            "plaquette": round(float(plaq_arr.mean()), 6),
            "plaquette_err": round(float(plaq_arr.std(ddof=1) / np.sqrt(n_meas)), 6),
            "sigma_a2": round(float(sigma_a2), 5) if np.isfinite(sigma_a2) else None,
            "sigma_a2_err": round(float(sigma_err), 5) if np.isfinite(sigma_err) else None,
            "coulomb_e": round(float(popt[2]), 4) if np.isfinite(popt[2]) else None
        },
        "artifact": {"file": fn_data, "size_mb": round(size_mb, 2), "sha256": data_hash}
    }

    with open(fn_manifest, 'w', encoding='utf-8') as f:
        json.dump(manifest, f, indent=4, ensure_ascii=False)
    manifest_hash = get_sha256(fn_manifest)

    print("\n" + "="*75)
    print(f"ИТОГОВЫЕ ФИЗИЧЕСКИЕ РЕЗУЛЬТАТЫ (BETA = {beta}, L = {L})")
    print("="*75)
    print(f"  <P> (Wilson Plaquette):   {plaq_arr.mean():.6f} +/- {plaq_arr.std(ddof=1)/np.sqrt(n_meas):.6f}")
    print(f"  String Tension a^2*sigma: {sigma_a2:.5f} +/- {sigma_err:.5f}")
    print(f"  Coulomb parameter e:      {popt[2]:.4f} (Lüscher ref: pi/12 = 0.2618)")
    print("-" * 75)
    print("  GEVP Stability Spectrum:")
    print(f"    Primary Hybrid   [0,4,8] (t0=1->t=2): m*a = {m_hyb_12:.4f} +/- {m12_err:.4f}  |  R_0 = {R_0_12:.3f} +/- {R_0_12_err:.3f}")
    print(f"    Primary Hybrid   [0,4,8] (t0=2->t=3): m*a = {m_hyb_23:.4f} +/- {m23_err:.4f}  |  R_0 = {R_0_23:.3f} +/- {R_0_23_err:.3f}")
    print(f"    Pure Smearing    [0,3,6] (t0=1->t=2): m*a = {m_sm_12:.4f} | cond={cond_sm:.1f}")
    print(f"    Pure Geometry    [6,7,8] (t0=1->t=2): m*a = {m_gm_12:.4f} | cond={cond_gm:.1f}")
    print("="*75)
    print(f"  DATA SHA-256:     {data_hash}")
    print(f"  MANIFEST SHA-256: {manifest_hash}")
    print("="*75)

if __name__ == "__main__":
    run_point_beta24_L16()

In [ ]:
import numpy as np
from scipy.optimize import curve_fit

fn = 'su2_production_beta24_L16.npz'
data = np.load(fn)
O_data = data['O_data']
W_data = data['W_data']
plaq_arr = data['plaq_arr']

L = 16
n_meas = O_data.shape[0]
n_bins = 40
bin_size = n_meas // n_bins
n_use = n_bins * bin_size

# 1. Плакетка
plaq_mean = plaq_arr.mean()
plaq_err = plaq_arr.std(ddof=1) / np.sqrt(n_meas)

# 2. Натяжение струны
R_values = (1, 2, 3, 4, 5)
T_values = (1, 2, 3, 4, 5)
loop_keys = [(R, T) for R in R_values for T in T_values]
W_binned = W_data[:n_use].reshape(n_bins, bin_size, len(loop_keys)).mean(axis=1)
W_mean = W_binned.mean(0)
W_map = {k: W_mean[ki] for ki, k in enumerate(loop_keys)}

def safe_log(a, b):
    return np.log(a / b) if (a > 0 and b > 0) else np.nan

def cornell(R, V0, sigma, e):
    return V0 + sigma * R - e / R

R_arr = np.array(R_values)
V_T3 = np.array([safe_log(W_map[(R, 3)], W_map[(R, 4)]) for R in R_values])
mask = (R_arr >= 2) & np.isfinite(V_T3)
popt, _ = curve_fit(cornell, R_arr[mask], V_T3[mask], p0=[0.53, 0.082, 0.26], bounds=([-2.0, 0.03, 0.0], [3.0, 0.25, 1.0]))
sigma_a2 = popt[1]

sigma_jk = np.full(n_bins, np.nan)
for b in range(n_bins):
    W_sub = np.delete(W_binned, b, axis=0).mean(axis=0)
    W_m = {k: W_sub[ki] for ki, k in enumerate(loop_keys)}
    Vb = np.array([safe_log(W_m[(R, 3)], W_m[(R, 4)]) for R in R_values])
    try:
        popt_b, _ = curve_fit(cornell, R_arr[mask], Vb[mask], p0=[0.53, 0.082, 0.26], bounds=([-2.0, 0.03, 0.0], [3.0, 0.25, 1.0]))
        sigma_jk[b] = popt_b[1]
    except:
        pass
sigma_err = np.sqrt((n_bins - 1) / n_bins * np.sum((sigma_jk - np.nanmean(sigma_jk))**2))

# 3. GEVP (Базис [0, 4, 8])
n_ops = O_data.shape[1]
G_cfg = np.zeros((n_meas, n_ops, n_ops, L))
for dt in range(L):
    G_cfg[:, :, :, dt] = np.einsum('mit,mjt->mij', O_data, np.roll(O_data, -dt, axis=-1)) / L
Om = O_data.mean(axis=-1)
Gb = G_cfg[:n_use].reshape(n_bins, bin_size, n_ops, n_ops, L).mean(axis=1)
Ob = Om[:n_use].reshape(n_bins, bin_size, n_ops).mean(axis=1)

def build_C(G, Om):
    C = G - np.outer(Om, Om)[:, :, None]
    C = 0.5 * (C + C.transpose(1, 0, 2))
    Cs = np.zeros_like(C)
    Cs[:, :, 0] = C[:, :, 0]
    for dt in range(1, L):
        Cs[:, :, dt] = 0.5 * (C[:, :, dt] + C[:, :, L-dt])
    return Cs

def gevp(C_mat, t0, t_tgt):
    C0 = 0.5 * (C_mat[:, :, t0] + C_mat[:, :, t0].T)
    w0, v0 = np.linalg.eigh(C0)
    good = w0 > (1e-3 * w0.max())
    if good.sum() < 2: return np.nan
    inv = v0[:, good] @ np.diag(1.0 / np.sqrt(w0[good]))
    Ct = 0.5 * (C_mat[:, :, t_tgt] + C_mat[:, :, t_tgt].T)
    w = np.linalg.eigvalsh(inv.T @ Ct @ inv)
    w = w[np.isfinite(w) & (w > 0)]
    return -np.log(w.max()) / (t_tgt - t0) if len(w) else np.nan

basis = [0, 4, 8]
C_full = build_C(Gb.mean(0), Ob.mean(0))[np.ix_(basis, basis)]
m12 = gevp(C_full, 1, 2)
m23 = gevp(C_full, 2, 3)

m12_jk = np.zeros(n_bins)
m23_jk = np.zeros(n_bins)
for b in range(n_bins):
    Cb = build_C(np.delete(Gb, b, axis=0).mean(0), np.delete(Ob, b, axis=0).mean(0))[np.ix_(basis, basis)]
    m12_jk[b] = gevp(Cb, 1, 2)
    m23_jk[b] = gevp(Cb, 2, 3)

def jk_e(a):
    v = a[np.isfinite(a)]
    return np.sqrt((len(v)-1)/len(v)*np.sum((v-v.mean())**2))

m12_err = jk_e(m12_jk)
m23_err = jk_e(m23_jk)

R12 = m12 / np.sqrt(sigma_a2)
R12_err = R12 * np.sqrt((m12_err/m12)**2 + 0.25*(sigma_err/sigma_a2)**2)
R23 = m23 / np.sqrt(sigma_a2)
R23_err = R23 * np.sqrt((m23_err/m23)**2 + 0.25*(sigma_err/sigma_a2)**2)

print("=" * 75)
print("ИТОГОВЫЕ ФИЗИЧЕСКИЕ РЕЗУЛЬТАТЫ (BETA = 2.4, L = 16)")
print("=" * 75)
print(f"  <P> (Wilson Plaquette):   {plaq_mean:.6f} +/- {plaq_err:.6f}")
print(f"  String Tension a^2*sigma: {sigma_a2:.5f} +/- {sigma_err:.5f}")
print(f"  Coulomb parameter e:      {popt[2]:.4f} (Lüscher ref: pi/12 = 0.2618)")
print("-" * 75)
print(f"  Primary Hybrid [0,4,8] (t0=1->t=2): m*a = {m12:.4f} +/- {m12_err:.4f}  |  R_0 = {R12:.3f} +/- {R12_err:.3f}")
print(f"  Primary Hybrid [0,4,8] (t0=2->t=3): m*a = {m23:.4f} +/- {m23_err:.4f}  |  R_0 = {R23:.3f} +/- {R23_err:.3f}")
print("=" * 75)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# 4 ансамбля (t=1->2)
x = np.array([0.14214, 0.08195, 0.08035, 0.04545])
y = np.array([3.258,   3.392,   3.356,   3.931])
yerr = np.array([0.213, 0.248,   0.226,   0.432])
labels = [r'$\beta=2.3\ (12^4)$', r'$\beta=2.4\ (12^4)$', r'$\beta=2.4\ (16^4)$', r'$\beta=2.5\ (16^4)$']

def linear(x, c0, c1):
    return c0 + c1 * x

popt, pcov = curve_fit(linear, x, y, sigma=yerr, absolute_sigma=True)
c0, c1 = popt
dc0, dc1 = np.sqrt(np.diag(pcov))

resid = y - linear(x, *popt)
chi2 = np.sum((resid / yerr)**2)
dof = len(x) - 2

print(f"R_0(0) = {c0:.3f} +/- {dc0:.3f}")
print(f"c_1    = {c1:.3f} +/- {dc1:.3f}")
print(f"chi2/dof = {chi2/dof:.2f}")

plt.figure(figsize=(7.5, 5.2), dpi=300)
xg = np.linspace(0, 0.16, 200)

# Полоса Тепера (1998)
plt.axhline(3.78, color='#d62728', linestyle='--', linewidth=1.5, label=r'Teper Benchmark: $3.78 \pm 0.16$')
plt.fill_between(xg, 3.78 - 0.16, 3.78 + 0.16, color='#d62728', alpha=0.10)

# Линейный фит Симанзика
plt.plot(xg, linear(xg, *popt), color='#1f77b4', linewidth=2.0, 
         label=rf'Symanzik Fit: $R_0(0) = {c0:.2f} \pm {dc0:.2f}\ (\chi^2/\mathrm{{dof}}={chi2/dof:.2f})$')
plt.fill_between(xg, linear(xg, *popt) - dc0, linear(xg, *popt) + dc0, color='#1f77b4', alpha=0.18, label=r'$1\sigma$ Confidence Band')

# Данные решётки
for i in range(len(x)):
    plt.errorbar(x[i], y[i], yerr=yerr[i], fmt='o', color='black', ecolor='black', 
                 elinewidth=1.6, capsize=4, capthick=1.2, markersize=6, zorder=5)
    plt.annotate(labels[i], (x[i], y[i]), textcoords="offset points", xytext=(8, -4), fontsize=9)

plt.xlabel(r'$a^2\sigma$', fontsize=12)
plt.ylabel(r'$m_{0^{++}} / \sqrt{\sigma}$', fontsize=12)
plt.title(r'SU(2) Scalar Glueball Continuum Extrapolation', fontsize=13, fontweight='bold', pad=12)
plt.xlim(0, 0.16)
plt.ylim(2.6, 4.6)
plt.grid(True, linestyle=':', alpha=0.5)
plt.legend(frameon=True, facecolor='white', framealpha=0.9, fontsize=9.5, loc='upper right')
plt.tight_layout()

plt.savefig('continuum_fit_4points.pdf')
plt.savefig('continuum_fit_4points.png')
plt.show()